# 02 — Build the Court document collection and structural relations

This notebook starts from the complete C- and T-prefixed case universe produced by notebook 01. It retrieves and flattens InfoCuria procedural metadata and **all documents associated with every case**, then derives two structural layers used in the released dataset:

1. a deterministic `proceeding_id` written directly onto the case table, grouping pilot and joined cases into one proceeding; and
2. a directed appeal table linking the case being appealed to the case in which the appeal is heard.

The notebook does **not** download source documents. Download selection and acquisition are handled in notebook 03. This separation keeps the source universe distinct from later analytical/acquisition decisions, which is useful for a Scientific Data Data Descriptor and for reproducibility.

## Inputs and outputs

**Input**

- `output/court_infocuria/court_cases_discovered.xlsx` — one row per discovered `procedureId` from notebook 01.

**Outputs**

- `output/court_infocuria/court_cases.xlsx` / `.csv` — enriched case universe with a derived `proceeding_id` layer.
- `output/court_infocuria/court_documents.xlsx` / `.csv` — complete InfoCuria document universe. No document rows are removed; `relevance_flag` records the project-specific document selection rule.
- `output/court_infocuria/court_appeals.xlsx` / `.csv` — directed ordinary appeal relationships, from the origin case to the appeal case.

Restartable API/cache state remains under `logs/court_infocuria/` and is working provenance rather than an additional analytical table.

In [1]:
# ---------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------
import json
import os
import re
import time
import asyncio
import logging
import random
from pathlib import Path
from datetime import datetime
from collections import Counter
from typing import Any
from urllib.parse import urlparse, quote

import pandas as pd
import numpy as np
import requests
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("enrich_infocuria_document_layer")


## 1. Paths, metadata settings, and document relevance definition

`procedureId` remains the canonical case key. Source identifiers from InfoCuria are retained alongside all derived identifiers.

The document universe is deliberately exhaustive. The `relevance_flag` is a **project-specific classification**, not a deletion rule. Under the current definition, the included document types are `ARRET`, `ARRET_SOM`, all `ARRET_DR` variants, `RECT`, and `DECISION`. `Arrêt (Information)` / `INF`, orders, Official Journal notices, extracts, opinions and other document types remain in the universe but are not selected for the text corpus.

In [2]:
# Repository paths

def find_project_root(start: Path | None = None) -> Path:
    override = os.environ.get("ECCJEU_PROJECT_ROOT", "").strip()
    if override:
        root = Path(override).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(f"ECCJEU_PROJECT_ROOT does not exist: {root}")
        return root
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "code" / "02_court").exists():
            return candidate
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the ec_cjeu project root. Run the notebook from within "
        "the repository or set ECCJEU_PROJECT_ROOT."
    )

PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / "output" / "court_infocuria"
LOG_DIR = PROJECT_ROOT / "logs" / "court_infocuria"
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "court_infocuria"
HTML_DIR = RAW_DIR / "html"
PDF_DIR = RAW_DIR / "pdf"

for directory in [OUTPUT_DIR, LOG_DIR, HTML_DIR, PDF_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

COURT_CASE_INPUT = OUTPUT_DIR / "court_cases_discovered.xlsx"
if not COURT_CASE_INPUT.exists():
    raise FileNotFoundError(f"Run notebook 01 first: {COURT_CASE_INPUT}")

CASE_METADATA_XLSX = OUTPUT_DIR / "court_cases.xlsx"
DOCUMENT_METADATA_XLSX = OUTPUT_DIR / "court_documents.xlsx"
APPEAL_METADATA_XLSX = OUTPUT_DIR / "court_appeals.xlsx"
CASE_METADATA_CSV = OUTPUT_DIR / "court_cases.csv"
DOCUMENT_METADATA_CSV = OUTPUT_DIR / "court_documents.csv"
APPEAL_METADATA_CSV = OUTPUT_DIR / "court_appeals.csv"

# Resumable acquisition state. JSONL is kept in logs only.
RAW_METADATA_JSONL = LOG_DIR / "procedural_metadata_state.jsonl"
JOINED_CASE_JSONL = LOG_DIR / "joined_case_state.jsonl"
DOWNLOAD_MANIFEST_JSONL = LOG_DIR / "document_download_state.jsonl"
FLATTEN_FAILED_JSONL = LOG_DIR / "metadata_flatten_failures.jsonl"

# Metadata acquisition mode
SCRAPE_MODE = "update_scrape"
VALID_SCRAPE_MODES = {"fresh_scrape", "update_scrape", "rebuild_from_cache"}
if SCRAPE_MODE not in VALID_SCRAPE_MODES:
    raise ValueError(f"Unknown SCRAPE_MODE: {SCRAPE_MODE}")
LIMIT_METADATA_FETCH_CASES = None
UPDATE_EXISTING_CACHE_RECORDS = False

PROCEDURES_URL = "https://infocuriaws.curia.europa.eu/elastic-connector/affairId/procedures"
LANGUAGE = "EN"
METADATA_REQUEST_TIMEOUT_SECONDS = 60
METADATA_MAX_RETRIES = 5
METADATA_SLEEP_SECONDS = 0.25

RELEVANT_DOC_TYPE_CODES = {
    "ARRET",
    "ARRET_SOM",
    "ARRET_DR",
    "RECT",
    "DECISION",
}
RELEVANCE_REASON = {
    "ARRET": "relevant_judgment",
    "ARRET_SOM": "relevant_judgment_summary",
    "ARRET_DR": "relevant_judgment_dr_variant",
    "RECT": "relevant_rectification_corrigendum",
    "DECISION": "relevant_reexamination_decision",
}

RUN_DOWNLOADS = True
OVERWRITE_EXISTING_FILES = False
DOWNLOAD_LIMIT = None
DOWNLOAD_SAVE_EVERY = 100
PREFERRED_LANGUAGES = ["EN", "DE", "FR"]
PREFERRED_FORMATS = ["HTML", "PDF"]
REQUEST_TIMEOUT_SECONDS = 90
DOWNLOAD_SLEEP_SECONDS = 1.25
DOWNLOAD_SLEEP_JITTER_SECONDS = 0.75
RETRIES_PER_URL = 3
RETRY_BACKOFF_SECONDS = 4.0
SESSION_WARMUP = True
SESSION_WARMUP_SLEEP_SECONDS = 0.75
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/126 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml,application/pdf,application/json;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9,de;q=0.8,fr;q=0.7",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Origin": "https://infocuria.curia.europa.eu",
    "Referer": "https://infocuria.curia.europa.eu/",
}

STREAM_JSONL_CHUNK_SIZE = 5000
KEEP_RAW_CONTENT_JSON = False
DROP_NON_ENGLISH_METADATA_COLUMNS = [
    "case_name_de", "case_name_fr", "case_object_de", "case_object_fr",
    "subject_matter_de", "subject_matter_fr", "procedure_result_de", "procedure_result_fr",
]
DROP_HUGE_DOWNLOAD_COLUMNS = ["candidate_download_records", "all_candidate_download_urls"]

# Joined-proceeding repair from the InfoCuria procedures endpoint.
RUN_JOINED_CASE_SEARCH = True
JOINED_CASE_SEARCH_LIMIT = None
JOINED_CASE_SLEEP_SECONDS = 0.15
JOINED_CASE_JSONL_MODE = "rebuild_jsonl"
VALID_JOINED_CASE_JSONL_MODES = {"rebuild_jsonl", "update_failed", "csv_from_jsonl"}
if JOINED_CASE_JSONL_MODE not in VALID_JOINED_CASE_JSONL_MODES:
    raise ValueError(f"Unknown JOINED_CASE_JSONL_MODE: {JOINED_CASE_JSONL_MODE}")
JOINED_CASE_RETRY_HTTP_STATUSES = {403, 429, 500, 502, 503, 504}
JOINED_CASE_BATCH_PAUSE_EVERY = 1000
JOINED_CASE_BATCH_PAUSE_SECONDS = 60
JOINED_CASE_COMPACT_JSONL_AFTER_RUN = True
JOINED_CASE_RECORD_RESPONSE_SUMMARY = True

COURT_INFOCURIA_RAW_DIR = RAW_DIR
COURT_INFOCURIA_HTML_DIR = HTML_DIR
COURT_INFOCURIA_PDFS_DIR = PDF_DIR
COURT_INFOCURIA_HTML_EXISTING_DIRS = [HTML_DIR]
COURT_INFOCURIA_PDF_EXISTING_DIRS = [PDF_DIR]

CASE_METADATA_STEM = "court_cases"
DOCUMENT_METADATA_STEM = "court_documents"
DOWNLOAD_MANIFEST_STEM = "court_document_downloads"


def read_manifest(path: Path) -> pd.DataFrame:
    path = Path(path)
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path, dtype=str)
    if path.suffix.lower() == ".jsonl":
        return pd.read_json(path, orient="records", lines=True, dtype=False)
    return pd.read_csv(path, dtype=str, low_memory=False)


def save_jsonl(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_json(path, orient="records", lines=True, force_ascii=False)


def save_table(df: pd.DataFrame, xlsx_path: Path, sheet_name: str) -> None:
    df.to_excel(xlsx_path, index=False, sheet_name=sheet_name[:31])


def file_size_mb(path: Path) -> float:
    return path.stat().st_size / (1024 * 1024)


def count_jsonl_lines(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open("rb") as handle:
        return sum(1 for _ in handle)

logger = logging.getLogger("court_infocuria_metadata")
logger.setLevel(logging.INFO)
if not logger.handlers:
    logger.addHandler(logging.StreamHandler())

print("Project root:", PROJECT_ROOT)
print("Case manifest:", COURT_CASE_INPUT)
print("Procedural metadata state:", RAW_METADATA_JSONL)

Project root: /home/edik/projects/ec_cjeu
Case manifest: /home/edik/projects/ec_cjeu/output/court_infocuria/court_cases_discovered.xlsx
Procedural metadata state: /home/edik/projects/ec_cjeu/logs/court_infocuria/procedural_metadata_state.jsonl


## 2. Read the discovered case universe

The case universe from notebook 01 is not substantively filtered here. Metadata requests are keyed and reconciled back to the source `procedureId`; readable case numbers and reconstructed keys are retained for diagnostics and human inspection.

In [3]:
df_cases_in = read_manifest(COURT_CASE_INPUT)
if df_cases_in.empty:
    raise ValueError("The InfoCuria case manifest is empty.")
required = {"procedureId", "affId", "case_number_raw", "internal_key"}
missing = sorted(required - set(df_cases_in.columns))
if missing:
    raise ValueError(f"Case manifest is missing required columns: {missing}")
print(f"Case procedures available: {len(df_cases_in):,}")
display(df_cases_in.head(5))

# ---------------------------------------------------------------------
# Generic helpers
# ---------------------------------------------------------------------

def clean_str(x):
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x).strip()



def first_nonempty(*values):
    """Return the first value that is not None/NA/empty after clean_str().

    This avoids pandas pd.NA boolean errors from expressions like
    `row.get("a") or row.get("b")`.
    """
    for value in values:
        s = clean_str(value)
        if s and s.lower() not in {"nan", "none", "<na>"}:
            return s
    return ""

def path_exists_safe(*values):
    """NA-safe check for the first non-empty path-like value."""
    s = first_nonempty(*values)
    if not s:
        return False
    try:
        return Path(s).exists()
    except Exception:
        return False

def normalize_whitespace(x):
    return re.sub(r"\s+", " ", clean_str(x)).strip()

def as_json_string(x):
    try:
        return json.dumps(x, ensure_ascii=False)
    except Exception:
        return "[]"

def unique_join(values, sep="|", max_items=None):
    vals = []
    for v in values:
        s = clean_str(v)
        if not s or s.lower() in {"nan", "none", "<na>"}:
            continue
        vals.append(s)
    vals = sorted(set(vals))
    if max_items is not None:
        vals = vals[:max_items]
    return sep.join(vals)

def ml_value(obj, lang="en", fallback=True):
    """Extract multilingual value from InfoCuria ML structures."""
    if isinstance(obj, str):
        try:
            obj = json.loads(obj)
        except Exception:
            return obj if fallback else ""
    lang = str(lang).lower()
    if isinstance(obj, list):
        found = []
        for item in obj:
            if isinstance(item, dict):
                for k, v in item.items():
                    if str(k).lower() == lang and clean_str(v):
                        return clean_str(v)
                    if clean_str(v):
                        found.append(clean_str(v))
        return found[0] if fallback and found else ""
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).lower() == lang and clean_str(v):
                return clean_str(v)
        vals = [clean_str(v) for v in obj.values() if clean_str(v)]
        return vals[0] if fallback and vals else ""
    return ""

def label_text(obj, lang="en"):
    """Extract label text from ML label structures."""
    if isinstance(obj, str):
        try:
            obj = json.loads(obj)
        except Exception:
            return obj
    if isinstance(obj, list):
        vals = []
        for item in obj:
            if isinstance(item, dict):
                label = item.get("label", item)
                val = ml_value(label, lang, fallback=True)
                if val:
                    vals.append(val)
        return "|".join(sorted(set(vals)))
    return ml_value(obj, lang, fallback=True)

def parse_affair_ref_list(values):
    """Pull published case IDs from InfoCuria affair reference fields."""
    if values is None:
        return []
    if isinstance(values, str):
        try:
            values = json.loads(values)
        except Exception:
            values = [values]
    if not isinstance(values, list):
        values = [values]
    out = []
    for v in values:
        if isinstance(v, dict):
            for key in ["publishedId", "publishedAffId", "affaire", "label", "value", "id"]:
                if clean_str(v.get(key)):
                    out.append(clean_str(v.get(key)))
                    break
        else:
            s = clean_str(v)
            # Often fields are like "C-123/20".
            m = re.search(r"\b[CT]\s*-?\s*\d{1,5}\s*/\s*\d{2,4}(?:\s+[A-Z][A-Z0-9-]*)?\b", s, flags=re.I)
            if m:
                out.append(normalize_whitespace(m.group(0)))
    return sorted(set(out))

def normalize_doc_type(value):
    return normalize_whitespace(value).upper().replace(" ", "_").replace("-", "_")

def split_doc_type_codes(value):
    """Return normalized document-type tokens.

    InfoCuria sometimes returns combined codes such as ORD_COMM|REQ_COMM.
    Relevance is evaluated token-wise, so ORD_COMM is recognised as
    relevant inside a combined code.
    """
    s = normalize_doc_type(value)
    if not s:
        return []
    return [p for p in re.split(r"[|,;/]+", s) if p]

def relevant_doc_type_tokens(code):
    return [t for t in split_doc_type_codes(code) if t in RELEVANT_DOC_TYPE_CODES]

def is_relevant_doc_type(code):
    return bool(relevant_doc_type_tokens(code))

def relevance_reason(code):
    hits = relevant_doc_type_tokens(code)
    if hits:
        return "|".join(RELEVANCE_REASON.get(h, f"relevant_{h.lower()}") for h in hits)
    return "irrelevant_non_final_or_duplicate_or_administrative_document"


# ---------------------------------------------------------------------
# Case-key helpers
# ---------------------------------------------------------------------

# - The upstream web/case manifest already contains `internal_key`.
# - Use that as the procedural key whenever present.
# - Only parse/generate keys as a fallback for older manifests.

def normalize_case_suffix(suffix):
    suffix = normalize_whitespace(suffix).upper()
    suffix = suffix.replace("–", "-").replace("—", "-")
    suffix = re.sub(r"[^A-Z0-9 -]+", "", suffix)
    suffix = re.sub(r"\s+", "-", suffix).strip("-")
    return suffix

def normalize_internal_key(value):
    """Normalize procedural key to YYYY/NNNN/PREFIX/SUFFIX/.

    Empty suffix is preserved as an empty fourth segment, e.g. 1966/0001/C//.
    This matches the upstream case skeleton internal_key format and prevents
    case/document join mismatches caused by 1966/0001/C/ vs 1966/0001/C//.
    """
    s = clean_str(value).upper().replace("\\", "/")
    if not s:
        return ""
    parts = s.split("/")
    # Preserve an explicitly empty suffix if present; remove only surrounding empty parts.
    if len(parts) >= 3 and parts[0] == "":
        parts = parts[1:]
    if len(parts) >= 4:
        year, num, prefix = parts[0], parts[1], parts[2]
        suffix = parts[3]
    else:
        compact = re.sub(r"/+", "/", s).strip("/")
        parts = compact.split("/") if compact else []
        if len(parts) >= 3:
            year, num, prefix = parts[0], parts[1], parts[2]
            suffix = parts[3] if len(parts) >= 4 else ""
        else:
            return re.sub(r"/+", "/", s if s.endswith("/") else s + "/")
    year = f"{int(year):04d}" if str(year).isdigit() else str(year)
    num = f"{int(num):04d}" if str(num).isdigit() else str(num)
    prefix = clean_str(prefix).upper()
    suffix = normalize_case_suffix(suffix)
    return f"{year}/{num}/{prefix}/{suffix}/"


def internal_key_base(value):
    s = normalize_internal_key(value)
    parts = [p for p in s.split("/") if p]
    if len(parts) >= 3:
        return f"{parts[0]}/{parts[1]}/{parts[2]}/"
    return ""

def make_slash_procedure_keys(prefix, num, year_full, suffix=""):
    prefix = clean_str(prefix).upper()
    suffix = normalize_case_suffix(suffix)
    if not prefix or num is None or not clean_str(year_full):
        return "", ""
    base = f"{int(year_full):04d}/{int(num):04d}/{prefix}/"
    full = f"{int(year_full):04d}/{int(num):04d}/{prefix}/{suffix}/"
    return full, base

def parse_case_number(raw: Any, clean: Any = None, prefix_hint: Any = None, year_hint: Any = None):
    raw_s = normalize_whitespace(raw).upper().replace("–", "-").replace("—", "-")
    clean_s = normalize_whitespace(clean).upper().replace("–", "-").replace("—", "-")
    prefix_hint = clean_str(prefix_hint).upper()

    slash_re = r"\b(\d{4})\s*/\s*(\d{1,5})\s*/\s*([CT])\s*/?\s*([A-Z0-9 -]*)\b"
    m0 = re.search(slash_re, raw_s) or (re.search(slash_re, clean_s) if clean_s else None)
    if m0:
        year_full = int(m0.group(1))
        num = int(m0.group(2))
        prefix = m0.group(3)
        suffix = normalize_case_suffix(m0.group(4))
        procedure_key, procedure_key_base = make_slash_procedure_keys(prefix, num, year_full, suffix)
        return {
            "case_prefix": prefix,
            "case_number_int": num,
            "case_number_int_full": f"{num:04d}",
            "case_number_padded": f"{num:04d}",
            "case_year_raw": str(year_full),
            "case_year_full": str(year_full),
            "case_suffix": suffix,
            "procedure_key": procedure_key,
            "procedure_key_base": procedure_key_base,
        }

    m = re.search(r"\b([CT])\s*-?\s*(\d{1,5})\s*/\s*(\d{2,4})\b\s*(.*)$", raw_s)
    if not m and clean_s:
        m = re.search(r"\b([CT])\s*-?\s*(\d{1,5})\s*/\s*(\d{2,4})\b\s*(.*)$", clean_s)

    if m:
        prefix = m.group(1)
        num = int(m.group(2))
        year_raw = m.group(3)
        suffix = normalize_case_suffix(m.group(4))
    else:
        m2 = re.search(r"\b(\d{1,5})\s*/\s*(\d{2,4})\b\s*(.*)$", raw_s or clean_s)
        if not m2:
            return {
                "case_prefix": prefix_hint or "",
                "case_number_int": None,
                "case_number_int_full": "",
                "case_number_padded": "",
                "case_year_raw": clean_str(year_hint),
                "case_year_full": "",
                "case_suffix": "",
                "procedure_key": "",
                "procedure_key_base": "",
            }
        prefix = prefix_hint or "C"
        num = int(m2.group(1))
        year_raw = m2.group(2)
        suffix = normalize_case_suffix(m2.group(3))

    year_int = int(year_raw)
    year_full = 1900 + year_int if year_int < 100 and year_int >= 50 else 2000 + year_int if year_int < 100 else year_int
    procedure_key, procedure_key_base = make_slash_procedure_keys(prefix, num, year_full, suffix)
    return {
        "case_prefix": prefix,
        "case_number_int": num,
        "case_number_int_full": f"{num:04d}",
        "case_number_padded": f"{num:04d}",
        "case_year_raw": year_raw,
        "case_year_full": str(year_full),
        "case_suffix": suffix,
        "procedure_key": procedure_key,
        "procedure_key_base": procedure_key_base,
    }

def add_or_preserve_procedure_key_columns(df, raw_col=None, clean_col=None, prefix_col=None, year_col=None, prefer_internal_key=True):
    if df.empty:
        return df.copy()
    out = df.copy()

    internal_col = next((c for c in ["internal_key", "procedure_key"] if c in out.columns), None) if prefer_internal_key else None
    has_internal = internal_col is not None and out[internal_col].fillna("").astype(str).str.strip().ne("")

    parsed_rows = []
    if raw_col is None:
        raw_col = next((c for c in ["case_number_raw", "publishedId", "case_number", "case", "affaire", "procedure_key"] if c in out.columns), None)
    if clean_col is None:
        clean_col = next((c for c in ["case_number_clean", "case_number", "procedure_key_base"] if c in out.columns), None)
    if prefix_col is None:
        prefix_col = next((c for c in ["case_prefix", "prefix"] if c in out.columns), None)
    if year_col is None:
        year_col = next((c for c in ["case_year_full", "year"] if c in out.columns), None)

    for _, row in out.iterrows():
        parsed_rows.append(parse_case_number(
            row.get(raw_col) if raw_col else "",
            row.get(clean_col) if clean_col else "",
            row.get(prefix_col) if prefix_col else "",
            row.get(year_col) if year_col else "",
        ))
    parsed_df = pd.DataFrame(parsed_rows, index=out.index)

    # Fill missing key component columns, but do not overwrite an upstream internal_key.
    for col in ["case_prefix", "case_number_int", "case_number_int_full", "case_number_padded", "case_year_raw", "case_year_full", "case_suffix"]:
        if col not in out.columns:
            out[col] = parsed_df.get(col)
        else:
            out[col] = out[col].where(out[col].fillna("").astype(str).str.strip().ne(""), parsed_df.get(col))

    if internal_col and prefer_internal_key:
        out["procedure_key"] = out[internal_col].map(normalize_internal_key)
        out["procedure_key_base"] = out["procedure_key"].map(internal_key_base)
    else:
        out["procedure_key"] = parsed_df["procedure_key"]
        out["procedure_key_base"] = parsed_df["procedure_key_base"]

    # Keep `internal_key` explicit in outputs.
    if "internal_key" not in out.columns:
        out["internal_key"] = out["procedure_key"]

    return out

df_cases = add_or_preserve_procedure_key_columns(
    df_cases_in,
    raw_col="case_number_raw" if "case_number_raw" in df_cases_in.columns else None,
    prefer_internal_key=True,
)

print("InfoCuria cases keyed:", df_cases.shape)
if not df_cases.empty and "procedure_key" in df_cases.columns:
    display(df_cases[[c for c in ["internal_key", "case_number_raw", "case_number_clean", "publishedId", "affId", "procedureId", "procedure_key", "procedure_key_base"] if c in df_cases.columns]].head(10))


Case procedures available: 57,413


,case_number_raw,case_number_clean,case_prefix,case_suffix,case_number_int,case_number_padded,case_year_raw,case_year_full,internal_key,affId,publishedId,publishedAffId,procedureId,case_name_en,jurisdictionCode,natureCode,introductionDate,closeDate,affairStateCode,subject_matter_text,procLang,ecli,joinExist,joinAffairs
0,C-1/00,C-1/00,C,NaN,1,0001,00,2000,2000/0001/C//,C/0001/00/00000000RD/01,C-1/00,C-1/00,C/0001/00/00000000RD/01/P/01,Commission v France,C,NaN,2000-01-04,2001-12-13,CLOTPUB,NaN,NaN,NaN,0,[]
1,C-1/00 SA,C-1/00 SA,C,SA,1,0001,00,2000,2000/0001/C/SA/,C/0001/00/00000000SA/01,C-1/00 SA,C-1/00 SA,C/0001/00/00000000SA/01/P/01,Cotecna Inspection v Commission,C,NaN,2000-12-14,2001-05-29,CLOTPUB,NaN,NaN,NaN,0,[]
2,C-1/01 P,C-1/01 P,C,P,1,0001,01,2001,2001/0001/C/P/,C/0001/01/00000000PV/01,C-1/01 P,C-1/01 P,C/0001/01/00000000PV/01/P/01,Asia Motor France and Others v Commission,C,NaN,2001-01-03,2001-09-20,CLOTPUB,NaN,NaN,NaN,0,[]
3,C-1/02,C-1/02,C,NaN,1,0001,02,2002,2002/0001/C//,C/0001/02/00000000RP/01,C-1/02,C-1/02,C/0001/02/00000000RP/01/P/01,Borgmann,C,NaN,2002-01-07,2004-04-01,CLOTPUB,NaN,NaN,NaN,0,[]
4,C-1/02 SA,C-1/02 SA,C,SA,1,0001,02,2002,2002/0001/C/SA/,C/0001/02/00000000SA/01,C-1/02 SA,C-1/02 SA,C/0001/02/00000000SA/01/P/01,Antippas v Commission,C,NaN,2002-08-22,2003-03-27,CLOTPUB,NaN,NaN,NaN,0,[]


InfoCuria cases keyed: (57413, 27)


,internal_key,case_number_raw,case_number_clean,publishedId,affId,procedureId,procedure_key,procedure_key_base
0,2000/0001/C//,C-1/00,C-1/00,C-1/00,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,2000/0001/C//,2000/0001/C/
1,2000/0001/C/SA/,C-1/00 SA,C-1/00 SA,C-1/00 SA,C/0001/00/00000000SA/01,C/0001/00/00000000SA/01/P/01,2000/0001/C/SA/,2000/0001/C/
2,2001/0001/C/P/,C-1/01 P,C-1/01 P,C-1/01 P,C/0001/01/00000000PV/01,C/0001/01/00000000PV/01/P/01,2001/0001/C/P/,2001/0001/C/
3,2002/0001/C//,C-1/02,C-1/02,C-1/02,C/0001/02/00000000RP/01,C/0001/02/00000000RP/01/P/01,2002/0001/C//,2002/0001/C/
4,2002/0001/C/SA/,C-1/02 SA,C-1/02 SA,C-1/02 SA,C/0001/02/00000000SA/01,C/0001/02/00000000SA/01/P/01,2002/0001/C/SA/,2002/0001/C/
5,2003/0001/C//,C-1/03,C-1/03,C-1/03,C/0001/03/00000000RP/01,C/0001/03/00000000RP/01/P/01,2003/0001/C//,2003/0001/C/
6,2003/0001/C/SA/,C-1/03 SA,C-1/03 SA,C-1/03 SA,C/0001/03/00000000SA/01,C/0001/03/00000000SA/01/P/01,2003/0001/C/SA/,2003/0001/C/
7,2004/0001/C//,C-1/04,C-1/04,C-1/04,C/0001/04/00000000RP/01,C/0001/04/00000000RP/01/P/01,2004/0001/C//,2004/0001/C/
8,2004/0001/C/SA/,C-1/04 SA,C-1/04 SA,C-1/04 SA,C/0001/04/00000000SA/01,C/0001/04/00000000SA/01/P/01,2004/0001/C/SA/,2004/0001/C/
9,2005/0001/C//,C-1/05,C-1/05,C-1/05,C/0001/05/00000000RP/01,C/0001/05/00000000RP/01/P/01,2005/0001/C//,2005/0001/C/


## 3. Retrieve procedural metadata

The InfoCuria procedures endpoint is routed using `affId`, while the returned procedural record is matched back to the requested `procedureId`. Responses are cached as JSONL so long runs can be resumed without changing the analytical output definition.

In [4]:
# ---------------------------------------------------------------------
# Optional affairId/procedures metadata fetch/cache construction
# ---------------------------------------------------------------------

def metadata_post_json(url, payload, max_retries=METADATA_MAX_RETRIES):
    last_error = None
    metadata_headers = {
        **HEADERS,
        "accept": "application/json",
        "content-type": "application/json; charset=UTF-8",
    }
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(
                url,
                headers=metadata_headers,
                json=payload,
                timeout=METADATA_REQUEST_TIMEOUT_SECONDS,
            )
            if response.status_code >= 400:
                last_error = f"HTTP {response.status_code}: {response.text[:500]}"
                raise RuntimeError(last_error)
            return response.json()
        except Exception as e:
            last_error = repr(e)
            wait = min(60, 2 ** attempt) + random.uniform(0, 1.0)
            logger.warning("Metadata request failed attempt %s/%s: %s | sleeping %.1fs", attempt, max_retries, last_error, wait)
            time.sleep(wait)
    raise RuntimeError(f"Metadata request failed after {max_retries} retries: {last_error}")



def fetch_procedure_record(
    aff_id: str,
    published_id: str = "",
    language: str = LANGUAGE,
):
    """
    Query the affairId/procedures endpoint for one procedure-manifest row.

    The endpoint is routed by affId, while the exact procedure label is passed
    in searchTerm. The returned response is later filtered to the requested
    procedureId before case metadata and documents are flattened.
    """
    payload = {
        "affId": aff_id,
        "searchTerm": f'"{published_id}"' if published_id else "",
        "tabName": "affair",
        "language": language,
        "advancedFiltersValue": [],
    }
    return metadata_post_json(PROCEDURES_URL, payload)


def append_raw_jsonl(path: Path, record: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8", newline="\n") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def iter_raw_jsonl(path: Path):
    if not path.exists():
        return
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except Exception as e:
                logger.warning("Skipping bad JSONL line %s: %r", line_no, e)


def raw_jsonl_key(record: dict) -> str:
    """
    procedureId is the canonical cache key.

    Fallbacks are retained only for rows where procedureId
    is absent.
    """
    return first_nonempty(
        record.get("procedureId"),
        record.get("internal_key"),
        record.get("procedure_key"),
        record.get("publishedId"),
        record.get("affId"),
    )


def compact_raw_jsonl_keep_last(path: Path):
    """Rewrite the JSONL so each procedure key appears once, keeping the last record."""
    if not path.exists():
        return

    latest = {}
    order = []

    for rec in iter_raw_jsonl(path):
        key = raw_jsonl_key(rec)
        if not key:
            key = f"__no_key__{len(order)}"

        if key not in latest:
            order.append(key)

        latest[key] = rec

    tmp = path.with_suffix(path.suffix + ".tmp")

    with open(tmp, "w", encoding="utf-8", newline="\n") as f:
        for key in order:
            f.write(json.dumps(latest[key], ensure_ascii=False) + "\n")

    tmp.replace(path)
    print(
        f"Compacted raw JSONL to {len(latest):,} unique procedure records "
        f"-> {path}"
    )


def load_raw_jsonl_keys(path: Path) -> set:
    keys = set()

    if not path.exists():
        return keys

    for obj in tqdm(
        iter_raw_jsonl(path),
        desc="Scanning procedure-level raw JSONL keys",
    ):
        key = raw_jsonl_key(obj)
        if key:
            keys.add(key)

    return keys


def prepare_procedure_fetch_spine(
    df_cases_source: pd.DataFrame,
) -> pd.DataFrame:
    """
    Produce one input row per procedureId.

    Rows with a valid procedureId are deduplicated only by procedureId.
    A fallback path is retained for rows missing procedureId so they can be
    logged and inspected rather than silently lost.
    """
    df_fetch = df_cases_source.copy()

    if "procedureId" not in df_fetch.columns:
        raise KeyError(
            "The InfoCuria case manifest must contain a procedureId column."
        )

    df_fetch["procedureId"] = (
        df_fetch["procedureId"]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

    has_procedure_id = df_fetch["procedureId"].notna()

    with_procedure_id = (
        df_fetch.loc[has_procedure_id]
        .drop_duplicates(subset=["procedureId"], keep="last")
    )

    without_procedure_id = df_fetch.loc[~has_procedure_id].copy()

    fallback_cols = [
        c
        for c in ["affId", "internal_key", "publishedId"]
        if c in without_procedure_id.columns
    ]

    if fallback_cols and not without_procedure_id.empty:
        without_procedure_id = without_procedure_id.drop_duplicates(
            subset=fallback_cols,
            keep="last",
        )

    df_fetch = pd.concat(
        [with_procedure_id, without_procedure_id],
        ignore_index=True,
    ).reset_index(drop=True)

    print(
        "Procedure-level fetch spine:",
        f"{len(df_fetch):,} rows |",
        f"{df_fetch['procedureId'].nunique(dropna=True):,} distinct procedureIds |",
        f"{df_fetch['procedureId'].isna().sum():,} rows missing procedureId",
    )

    return df_fetch


def build_or_update_metadata_jsonl(
    df_cases_source: pd.DataFrame,
):
    mode = SCRAPE_MODE

    if mode == "rebuild_from_cache":
        print(
            "SCRAPE_MODE='rebuild_from_cache': using existing "
            "procedure-level JSONL checkpoint only:",
            RAW_METADATA_JSONL,
        )
        return

    if mode == "fresh_scrape" and RAW_METADATA_JSONL.exists():
        RAW_METADATA_JSONL.unlink()
        print(
            "Deleted old procedure-level raw JSONL for fresh scrape:",
            RAW_METADATA_JSONL,
        )

    df_fetch = prepare_procedure_fetch_spine(df_cases_source)

    if LIMIT_METADATA_FETCH_CASES is not None:
        df_fetch = df_fetch.head(
            int(LIMIT_METADATA_FETCH_CASES)
        ).copy()

    raw_cache_keys = (
        load_raw_jsonl_keys(RAW_METADATA_JSONL)
        if RAW_METADATA_JSONL.exists()
        else set()
    )

    run_log_rows = []
    failed_rows = []

    for _, row in tqdm(
        df_fetch.iterrows(),
        total=len(df_fetch),
        desc=(
            f"{mode}: fetching InfoCuria procedure-level "
            "affairId/procedures metadata"
        ),
    ):
        procedure_id = clean_str(row.get("procedureId"))
        aff_id = clean_str(row.get("affId"))
        internal_key = clean_str(
            row.get("internal_key")
            or row.get("procedure_key")
        )
        published_id = clean_str(
            row.get("publishedId")
            or row.get("case_number_raw")
            or row.get("case_number_clean")
        )

        cache_key = first_nonempty(
            procedure_id,
            internal_key,
            published_id,
            aff_id,
        )

        if not procedure_id:
            failed_rows.append({
                "affId": aff_id,
                "procedureId": procedure_id,
                "internal_key": internal_key,
                "publishedId": published_id,
                "procedure_key": row.get("procedure_key"),
                "error": "missing_procedureId",
            })
            continue

        if not aff_id:
            failed_rows.append({
                "affId": aff_id,
                "procedureId": procedure_id,
                "internal_key": internal_key,
                "publishedId": published_id,
                "procedure_key": row.get("procedure_key"),
                "error": "missing_affId",
            })
            continue

        if (
            mode == "update_scrape"
            and cache_key in raw_cache_keys
            and not UPDATE_EXISTING_CACHE_RECORDS
        ):
            run_log_rows.append({
                "affId": aff_id,
                "procedureId": procedure_id,
                "internal_key": internal_key,
                "publishedId": published_id,
                "procedure_key": row.get("procedure_key"),
                "status": "cached_skipped",
            })
            continue

        try:
            data = fetch_procedure_record(
                aff_id=aff_id,
                published_id=published_id,
                language=LANGUAGE,
            )

            rec = {
                "fetched_at": (
                    datetime.utcnow()
                    .isoformat(timespec="seconds")
                    + "Z"
                ),
                "cache_key": cache_key,
                "affId": aff_id,
                "procedureId": procedure_id,
                "internal_key": internal_key,
                "publishedId": published_id,
                "procedure_key": row.get("procedure_key"),
                "response": data,
            }

            append_raw_jsonl(
                RAW_METADATA_JSONL,
                rec,
            )

            raw_cache_keys.add(cache_key)

            run_log_rows.append({
                "affId": aff_id,
                "procedureId": procedure_id,
                "internal_key": internal_key,
                "publishedId": published_id,
                "procedure_key": row.get("procedure_key"),
                "status": "ok",
                "totalHits": data.get("totalHits"),
            })

        except Exception as e:
            failed_rows.append({
                "affId": aff_id,
                "procedureId": procedure_id,
                "internal_key": internal_key,
                "publishedId": published_id,
                "procedure_key": row.get("procedure_key"),
                "error": repr(e),
            })

            run_log_rows.append({
                "affId": aff_id,
                "procedureId": procedure_id,
                "internal_key": internal_key,
                "publishedId": published_id,
                "procedure_key": row.get("procedure_key"),
                "status": "failed",
                "error": repr(e),
            })

        time.sleep(float(METADATA_SLEEP_SECONDS))

    df_log = pd.DataFrame(run_log_rows)
    df_fail = pd.DataFrame(failed_rows)

    compact_raw_jsonl_keep_last(
        RAW_METADATA_JSONL
    )

    print("Procedure-level metadata fetch/update pass complete.")
    print("Fetch log rows:", f"{len(run_log_rows):,}")
    print("Failures:", f"{len(failed_rows):,}")


build_or_update_metadata_jsonl(df_cases)

if not RAW_METADATA_JSONL.exists():
    raise FileNotFoundError(f"Raw metadata JSONL still does not exist: {RAW_METADATA_JSONL}")

print("Raw JSONL ready:", RAW_METADATA_JSONL)
print("Raw JSONL size MB:", round(file_size_mb(RAW_METADATA_JSONL), 2))
print("Raw JSONL line count:", f"{count_jsonl_lines(RAW_METADATA_JSONL):,}")


Procedure-level fetch spine: 57,413 rows | 57,413 distinct procedureIds | 0 rows missing procedureId


Scanning procedure-level raw JSONL keys: 0it [00:00, ?it/s]

update_scrape: fetching InfoCuria procedure-level affairId/procedures metadata:   0%|          | 0/57413 [00:0…

Compacted raw JSONL to 57,413 unique procedure records -> /home/edik/projects/ec_cjeu/logs/court_infocuria/procedural_metadata_state.jsonl
Procedure-level metadata fetch/update pass complete.
Fetch log rows: 57,413
Failures: 0
Raw JSONL ready: /home/edik/projects/ec_cjeu/logs/court_infocuria/procedural_metadata_state.jsonl
Raw JSONL size MB: 45346.3
Raw JSONL line count: 57,413


## 4. Construct the complete case and document universes

Every document attached by InfoCuria to a case in the universe is retained. Document relevance is stored as metadata (`relevance_flag`, `relevance_category`, `relevance_reason`) rather than used to remove rows. This makes the raw source universe and the later corpus-selection decision separately auditable.

In [5]:
# ---------------------------------------------------------------------
# Raw JSONL iterator and InfoCuria flattening helpers
# ---------------------------------------------------------------------

def iter_raw_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except Exception as e:
                logger.warning("Skipping bad JSONL line %s: %r", line_no, e)

def select_requested_procedure_hit(raw_rec):
    """
    Select the response hit matching the requested procedureId.

    Matching order:
      1. exact procedureId
      2. exact publishedId
      3. exact affId only when the response contains one hit
      4. no match -> return an empty result rather than silently using
         the wrong procedure
    """
    response = raw_rec.get("response") or {}
    hits = response.get("searchHits") or []

    if not hits:
        return None, {}

    requested_procedure_id = clean_str(
        raw_rec.get("procedureId")
    )

    if requested_procedure_id:
        for hit in hits:
            content = hit.get("content") or {}

            if clean_str(
                content.get("procedureId")
            ) == requested_procedure_id:
                return hit, content

    requested_published_id = clean_str(
        raw_rec.get("publishedId")
    )

    if requested_published_id:
        for hit in hits:
            content = hit.get("content") or {}

            if clean_str(
                content.get("publishedId")
            ) == requested_published_id:
                return hit, content

    if len(hits) == 1:
        hit = hits[0]
        content = hit.get("content") or {}

        requested_aff_id = clean_str(
            raw_rec.get("affId")
        )

        if (
            not requested_aff_id
            or clean_str(content.get("affId")) == requested_aff_id
        ):
            return hit, content

    return None, {}

def extract_inner_document_hits(hit):
    if not isinstance(hit, dict):
        return []
    inner = hit.get("innerHits") or {}
    doc_block = inner.get("document") or {}
    return doc_block.get("searchHits") or []

def extract_available_langs_from_content_ml(content_ml):
    if isinstance(content_ml, str):
        try:
            content_ml = json.loads(content_ml)
        except Exception:
            return []
    langs = []
    if isinstance(content_ml, list):
        for item in content_ml:
            if isinstance(item, dict):
                langs.extend([str(k).upper() for k in item.keys()])
    return sorted(set(langs))

def normalize_doc_formats(value):
    if value is None:
        return []
    if isinstance(value, list):
        vals = value
    elif isinstance(value, str):
        try:
            parsed = json.loads(value)
            vals = parsed if isinstance(parsed, list) else [value]
        except Exception:
            vals = re.split(r"[|,; ]+", value)
    else:
        vals = [value]
    return sorted(set([str(x).upper().strip() for x in vals if str(x).strip()]))

# Build input lookup once. This enriches raw JSON rows with fields from the original input manifest.
INPUT_BY_AFFID = {}
INPUT_BY_PROCEDURE_ID = {}
INPUT_BY_PUBLISHED_ID = {}

if not df_cases.empty:
    for _, _row in df_cases.iterrows():
        d = _row.to_dict()
        for col, target in [
            ("affId", INPUT_BY_AFFID),
            ("procedureId", INPUT_BY_PROCEDURE_ID),
            ("publishedId", INPUT_BY_PUBLISHED_ID),
            ("case_number_raw", INPUT_BY_PUBLISHED_ID),
        ]:
            key = clean_str(_row.get(col)) if col in df_cases.columns else ""
            if key and key not in target:
                target[key] = d

def lookup_input_row(raw_rec, content):
    """
    Resolve the upstream manifest row using procedureId first.
    """
    for key, mapping in [
        (
            clean_str(
                raw_rec.get("procedureId")
                or content.get("procedureId")
            ),
            INPUT_BY_PROCEDURE_ID,
        ),
        (
            clean_str(
                raw_rec.get("publishedId")
                or content.get("publishedId")
            ),
            INPUT_BY_PUBLISHED_ID,
        ),
        (
            clean_str(
                raw_rec.get("affId")
                or content.get("affId")
            ),
            INPUT_BY_AFFID,
        ),
    ]:
        if key and key in mapping:
            return mapping[key]

    return {}

def flatten_case_record(raw_rec):
    response = raw_rec.get("response") or {}
    hit, c = select_requested_procedure_hit(raw_rec)

    if not c:
        raise ValueError(
            "No response hit matched requested procedureId="
            f"{raw_rec.get('procedureId')!r}, "
            f"publishedId={raw_rec.get('publishedId')!r}, "
            f"affId={raw_rec.get('affId')!r}"
        )

    input_row = lookup_input_row(raw_rec, c)

    published = c.get("publishedId") or input_row.get("publishedId") or input_row.get("case_number_raw") or raw_rec.get("publishedId")
    parsed = parse_case_number(published, input_row.get("case_number_clean"), input_row.get("case_prefix"))

    calendars = c.get("calendars") or []
    delivery_dates = sorted({x.get("seanceDate") for x in calendars if isinstance(x, dict) and x.get("seanceDate")})
    delivery_types = sorted({x.get("seanceType") for x in calendars if isinstance(x, dict) and x.get("seanceType")})

    # Keep the existing notebook-level procedure_key for compatibility, but also expose
    # an explicit InfoCuria slash key in the requested format:
    #   infocuria_slash_procedure_key_base = YYYY/NNNN/PREFIX/
    #   infocuria_slash_procedure_key      = YYYY/NNNN/PREFIX/ or YYYY/NNNN/PREFIX/SUFFIX
    # This makes the matching key auditable even if upstream InfoCuria metadata already
    # contains another field called procedure_key.
    infocuria_slash_procedure_key = parsed.get("procedure_key") or ""
    infocuria_slash_procedure_key_base = parsed.get("procedure_key_base") or ""

    row = {
        "internal_key": normalize_internal_key(input_row.get("internal_key") or raw_rec.get("internal_key") or raw_rec.get("procedure_key") or input_row.get("procedure_key") or infocuria_slash_procedure_key),
        "procedure_key": normalize_internal_key(input_row.get("internal_key") or raw_rec.get("internal_key") or raw_rec.get("procedure_key") or input_row.get("procedure_key") or infocuria_slash_procedure_key),
        "procedure_key_base": internal_key_base(input_row.get("internal_key") or raw_rec.get("internal_key") or raw_rec.get("procedure_key") or input_row.get("procedure_key") or infocuria_slash_procedure_key) or infocuria_slash_procedure_key_base or input_row.get("procedure_key_base"),
        "infocuria_slash_procedure_key": infocuria_slash_procedure_key,
        "infocuria_slash_procedure_key_base": infocuria_slash_procedure_key_base,
        "case_number_raw": published,
        "case_number_clean": input_row.get("case_number_clean"),
        "case_prefix": parsed.get("case_prefix"),
        "case_number_int": parsed.get("case_number_int"),
        "case_number_int_full": parsed.get("case_number_int_full"),
        "case_year_raw": parsed.get("case_year_raw"),
        "case_year_full": parsed.get("case_year_full"),
        "case_suffix": parsed.get("case_suffix"),
        "affId": c.get("affId") or raw_rec.get("affId") or input_row.get("affId"),
        "publishedId": c.get("publishedId") or published,
        "publishedAffId": c.get("publishedAffId"),
        "procedureId": c.get("procedureId") or raw_rec.get("procedureId") or input_row.get("procedureId"),
        "jurisdiction": c.get("jurisdiction"),
        "jurisdictionCode": c.get("jurisdictionCode"),
        "nature": c.get("nature"),
        "natureCode": c.get("natureCode"),
        "procType": c.get("procType"),
        "procNumber": c.get("procNumber"),
        "suiteNumber": c.get("suiteNumber"),
        "case_name_en": ml_value(c.get("usualNameML"), "en"),

        "case_object_en": ml_value(c.get("affObjectML"), "en"),

        "introductionDate": c.get("introductionDate"),
        "procIntroDate": c.get("procIntroDate"),
        "introductionYear": c.get("introductionYear"),
        "closeDate": c.get("closeDate"),
        "procClosDate": c.get("procClosDate"),
        "closeYear": c.get("closeYear"),
        "affairState": c.get("affairState"),
        "affairStateCode": c.get("affairStateCode"),
        "procState": c.get("procState"),
        "procLang": c.get("procLang"),
        "joinExist": c.get("joinExist"),
        "pilot": c.get("pilot"),
        "idPilot": c.get("idPilot"),
        "pilot_case_publishedId": parse_affair_ref_list([c.get("idPilot")])[0] if c.get("idPilot") and parse_affair_ref_list([c.get("idPilot")]) else "",
        "joinAffairs": as_json_string(c.get("joinAffairs") or []),
        "joinAffairs_publishedIds": as_json_string(parse_affair_ref_list(c.get("joinAffairs") or [])),
        "pourvoiAffIds": as_json_string(c.get("pourvoiAffIds") or []),
        "pourvoi_publishedIds": as_json_string(parse_affair_ref_list(c.get("pourvoiAffIds") or [])),
        "originPourvoiIds": as_json_string(c.get("originPourvoiIds") or []),
        "origin_pourvoi_publishedIds": as_json_string(parse_affair_ref_list(c.get("originPourvoiIds") or [])),
        "reExamenAffIds": as_json_string(c.get("reExamenAffIds") or []),
        "originReExamanIds": as_json_string(c.get("originReExamanIds") or []),
        "transfertAffIds": as_json_string(c.get("transfertAffIds") or []),
        "originTransfertIds": as_json_string(c.get("originTransfertIds") or []),
        "matCode": as_json_string(c.get("matCode") or []),
        "subject_matter_en": label_text(c.get("matCodeML"), "en"),

        "reportingJudge": c.get("reportingJudge"),
        "formation": c.get("formation"),
        "advocate_general_code": c.get("avg"),
        "resultTypes": as_json_string(c.get("resultTypes") or []),
        "procedure_result_en": label_text(c.get("procedureResultTypeML"), "en"),

        "parties": c.get("parties"),
        "conclsDate": as_json_string(c.get("conclsDate") or []),
        "conclLang": c.get("conclLang"),
        "delivery_dates": as_json_string(delivery_dates),
        "delivery_types": as_json_string(delivery_types),
        "numDocs": c.get("numDocs"),
        "doctrineNotes_count": len(c.get("doctrineNotes") or []),
        "lastModifiedDate": c.get("lastModifiedDate"),
        "totalHits": response.get("totalHits"),
        "inner_document_totalHits": (((hit or {}).get("innerHits") or {}).get("document") or {}).get("totalHits"),
        "raw_case_content_json": json.dumps(c, ensure_ascii=False) if KEEP_RAW_CONTENT_JSON else "",
    }
    return row

def flatten_document_records(raw_rec):
    response = raw_rec.get("response") or {}
    hit, c = select_requested_procedure_hit(raw_rec)

    if not c:
        raise ValueError(
            "No document-bearing response hit matched requested procedureId="
            f"{raw_rec.get('procedureId')!r}, "
            f"publishedId={raw_rec.get('publishedId')!r}, "
            f"affId={raw_rec.get('affId')!r}"
        )

    case_row = flatten_case_record(raw_rec)
    docs = extract_inner_document_hits(hit)
    rows = []

    for pos, dh in enumerate(docs):
        dc = dh.get("content") or {}
        content_ml = dc.get("contentML") or []
        available_langs = extract_available_langs_from_content_ml(content_ml)
        doc_formats = normalize_doc_formats(dc.get("docFormats") or ([dc.get("docFormat")] if dc.get("docFormat") else []))
        code = normalize_doc_type(dc.get("docTypeCode") or dc.get("docType"))

        row = {
            "internal_key": case_row.get("internal_key"),
            "procedure_key": case_row.get("procedure_key"),
            "procedure_key_base": case_row.get("procedure_key_base"),
            "infocuria_slash_procedure_key": case_row.get("infocuria_slash_procedure_key"),
            "infocuria_slash_procedure_key_base": case_row.get("infocuria_slash_procedure_key_base"),
            "case_number_raw": case_row.get("case_number_raw"),
            "case_number_clean": case_row.get("case_number_clean"),
            "case_prefix": case_row.get("case_prefix"),
            "case_number_int": case_row.get("case_number_int"),
            "case_number_int_full": case_row.get("case_number_int_full"),
            "case_year_full": case_row.get("case_year_full"),
            "case_suffix": case_row.get("case_suffix"),
            "affId": case_row.get("affId"),
            "publishedId": case_row.get("publishedId"),
            "procedureId": case_row.get("procedureId"),
            "procLang": case_row.get("procLang"),
            "case_name_en": case_row.get("case_name_en"),
            "introductionDate": case_row.get("introductionDate"),
            "closeDate": case_row.get("closeDate"),
            "procIntroDate": case_row.get("procIntroDate"),
            "procClosDate": case_row.get("procClosDate"),
            "doc_position": pos,
            "inner_hit_id": dh.get("id"),
            "docId": dc.get("docId"),
            "logicDocId": dc.get("logicDocId"),
            "jpLogicDocId": dc.get("jpLogicDocId"),
            "idProcedure": dc.get("idProcedure"),
            "idPublished": dc.get("idPublished"),
            "docType": dc.get("docType"),
            "docTypeCode": dc.get("docTypeCode"),
            "normalized_docTypeCode": code,
            "relevance_flag": is_relevant_doc_type(code),
            "relevance_category": "relevant" if is_relevant_doc_type(code) else "irrelevant",
            "relevance_reason": relevance_reason(code),
            "lnsDec": dc.get("lnsDec"),
            "numDoc": dc.get("numDoc"),
            "docDate": dc.get("docDate"),
            "docFormat": dc.get("docFormat"),
            "docFormats": as_json_string(doc_formats),
            "docLang": str(dc.get("docLang") or "").upper(),
            "available_languages": as_json_string(available_langs),
            "docUrl": dc.get("docUrl"),
            "contentLength": dc.get("contentLength"),
            # Keep only English context text here. It is split into a separate compact context manifest
            # before writing final outputs, so the metadata manifest remains manageable.
            "context_text_en": ml_value(content_ml, "en", fallback=False),
            "raw_document_content_json": json.dumps(dc, ensure_ascii=False) if KEEP_RAW_CONTENT_JSON else "",
        }

        row["document_dedup_key"] = "|".join([
            clean_str(row.get("procedureId"))
            or clean_str(row.get("procedure_key")),
            clean_str(row.get("logicDocId"))
            or clean_str(row.get("docId"))
            or clean_str(row.get("inner_hit_id")),
            clean_str(row.get("normalized_docTypeCode")),
            clean_str(row.get("docDate")),
        ])
        rows.append(row)

    return rows


# ---------------------------------------------------------------------
# Stream raw JSONL into flattened case/document tables
# ---------------------------------------------------------------------

# Stream files are written directly under LOG_DIR and deleted after loading.
CASE_METADATA_STREAM_JSONL = LOG_DIR / "_case_metadata_stream.jsonl"
DOCUMENT_METADATA_STREAM_JSONL = LOG_DIR / "_document_metadata_stream.jsonl"

for p in [CASE_METADATA_STREAM_JSONL, DOCUMENT_METADATA_STREAM_JSONL, FLATTEN_FAILED_JSONL]:
    if p.exists():
        p.unlink()

def append_rows_jsonl(rows, path: Path):
    if not rows:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8", newline="\n") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

case_buffer = []
doc_buffer = []
failed_buffer = []
seen_cases = 0
seen_docs = 0

for rec in tqdm(iter_raw_jsonl(RAW_METADATA_JSONL), desc="Streaming flatten raw JSONL"):
    try:
        case_buffer.append(flatten_case_record(rec))
        doc_rows = flatten_document_records(rec)
        doc_buffer.extend(doc_rows)
        seen_cases += 1
        seen_docs += len(doc_rows)
    except Exception as e:
        failed_buffer.append({
            "affId": rec.get("affId"),
            "publishedId": rec.get("publishedId"),
            "procedureId": rec.get("procedureId"),
            "internal_key": rec.get("internal_key"),
            "procedure_key": rec.get("procedure_key"),
            "error": "flatten_failed: " + repr(e),
        })

    if len(case_buffer) >= STREAM_JSONL_CHUNK_SIZE:
        append_rows_jsonl(case_buffer, CASE_METADATA_STREAM_JSONL)
        case_buffer = []
    if len(doc_buffer) >= STREAM_JSONL_CHUNK_SIZE:
        append_rows_jsonl(doc_buffer, DOCUMENT_METADATA_STREAM_JSONL)
        doc_buffer = []
    if len(failed_buffer) >= STREAM_JSONL_CHUNK_SIZE:
        append_rows_jsonl(failed_buffer, FLATTEN_FAILED_JSONL)
        failed_buffer = []

append_rows_jsonl(case_buffer, CASE_METADATA_STREAM_JSONL)
append_rows_jsonl(doc_buffer, DOCUMENT_METADATA_STREAM_JSONL)
append_rows_jsonl(failed_buffer, FLATTEN_FAILED_JSONL)

print("Streamed case records:", f"{seen_cases:,}")
print("Streamed document records:", f"{seen_docs:,}")
print("Flatten failures:", FLATTEN_FAILED_JSONL if FLATTEN_FAILED_JSONL.exists() else "none")

df_case_flat = read_manifest(CASE_METADATA_STREAM_JSONL) if CASE_METADATA_STREAM_JSONL.exists() else pd.DataFrame()
df_doc_meta = read_manifest(DOCUMENT_METADATA_STREAM_JSONL) if DOCUMENT_METADATA_STREAM_JSONL.exists() else pd.DataFrame()

# Deduplicate internally.
#
# procedureId is the canonical case/procedure key. Rows with a populated
# procedureId are deduplicated only by procedureId. Fallback identifiers
# are used only for rows where procedureId is missing.
if not df_case_flat.empty:
    if "procedureId" in df_case_flat.columns:
        has_procedure_id = (
            df_case_flat["procedureId"]
            .fillna("")
            .astype(str)
            .str.strip()
            .ne("")
        )

        with_procedure_id = (
            df_case_flat.loc[has_procedure_id]
            .drop_duplicates(subset=["procedureId"], keep="last")
        )

        without_procedure_id = df_case_flat.loc[~has_procedure_id].copy()

        fallback_cols = [
            c
            for c in ["affId", "procedure_key", "internal_key"]
            if c in without_procedure_id.columns
        ]

        if fallback_cols:
            without_procedure_id = without_procedure_id.drop_duplicates(
                subset=fallback_cols,
                keep="last",
            )

        df_case_flat = pd.concat(
            [with_procedure_id, without_procedure_id],
            ignore_index=True,
        ).reset_index(drop=True)

    else:
        fallback_cols = [
            c
            for c in ["affId", "procedure_key", "internal_key"]
            if c in df_case_flat.columns
        ]

        if fallback_cols:
            df_case_flat = df_case_flat.drop_duplicates(
                subset=fallback_cols,
                keep="last",
            ).reset_index(drop=True)

if not df_doc_meta.empty and "document_dedup_key" in df_doc_meta.columns:
    df_doc_meta = df_doc_meta.drop_duplicates(
        subset=["document_dedup_key"],
        keep="last",
    ).reset_index(drop=True)

# Remove large context/raw fields and non-English fields from document metadata.
large_doc_cols = [
    "context_text_en", "content_text_en", "content_text_de", "content_text_fr", "content_text_best", "raw_document_content_json"
]
df_doc_meta = df_doc_meta.drop(columns=[c for c in large_doc_cols + DROP_NON_ENGLISH_METADATA_COLUMNS if c in df_doc_meta.columns], errors="ignore")
df_case_flat = df_case_flat.drop(columns=[c for c in DROP_NON_ENGLISH_METADATA_COLUMNS if c in df_case_flat.columns], errors="ignore")

# Remove temporary stream files. Final manifests are written later.
for p in [CASE_METADATA_STREAM_JSONL, DOCUMENT_METADATA_STREAM_JSONL]:
    try:
        if p.exists():
            p.unlink()
    except Exception as e:
        print(f"Could not delete temporary stream file {p}: {e}")

print("Case metadata rows after dedupe:", df_case_flat.shape)
print("Document metadata rows after dedupe, without context text:", df_doc_meta.shape)
display(df_case_flat.head(3))
display(df_doc_meta.head(3))

# ---------------------------------------------------------------------
# Document metadata manifest base
# ---------------------------------------------------------------------

# The final document metadata manifest is df_doc_meta after relevance columns are assigned.
# It intentionally excludes context text and raw document JSON.
print("Document metadata base rows:", len(df_doc_meta))
if not df_doc_meta.empty:
    display(df_doc_meta.head(10))

# ---------------------------------------------------------------------
# 2) Filtered relevant document metadata manifest
# ---------------------------------------------------------------------

if df_doc_meta.empty:
    df_relevant_docs = df_doc_meta.copy()
else:
    df_doc_meta["normalized_docTypeCode"] = df_doc_meta["normalized_docTypeCode"].map(normalize_doc_type)
    df_doc_meta["relevance_flag"] = df_doc_meta["normalized_docTypeCode"].map(is_relevant_doc_type)
    df_doc_meta["relevance_category"] = df_doc_meta["relevance_flag"].map({True: "relevant", False: "irrelevant"})
    df_doc_meta["relevance_reason"] = df_doc_meta["normalized_docTypeCode"].map(relevance_reason)
    df_doc_meta["relevant_doc_type_tokens"] = df_doc_meta["normalized_docTypeCode"].map(lambda x: "|".join(relevant_doc_type_tokens(x)))

    df_relevant_docs = df_doc_meta[df_doc_meta["relevance_flag"]].copy().reset_index(drop=True)

print("Relevant document rows:", len(df_relevant_docs))
if not df_relevant_docs.empty:
    display(df_relevant_docs[[
        c for c in ["procedure_key", "procedure_key_base", "infocuria_slash_procedure_key", "infocuria_slash_procedure_key_base", "publishedId", "logicDocId", "docType", "docTypeCode", "docDate", "available_languages", "docFormats", "relevance_reason"]
        if c in df_relevant_docs.columns
    ]].head(20))


# ---------------------------------------------------------------------
# 3) Case-level document availability aggregation and case metadata join
# ---------------------------------------------------------------------

def bool_series(s):
    return s.astype(str).str.lower().isin(["true", "1", "yes"])

def normalized_date_join(values, sep="|"):
    vals = []
    for v in values:
        s = clean_str(v)
        if not s or s.lower() in {"nan", "none", "<na>"}:
            continue
        try:
            dt = pd.to_datetime(s, errors="coerce", dayfirst=False)
            vals.append(dt.strftime("%Y-%m-%d") if pd.notna(dt) else s)
        except Exception:
            vals.append(s)
    return sep.join(sorted(set(vals)))

def best_join_key_cols(left, right):
    """
    Return the best shared identifier for joining two tables.

    procedureId is the canonical key. affId is only a fallback when
    procedureId is unavailable in either table.
    """
    for cols in [
        ["procedureId"],
        ["affId"],
        ["internal_key"],
        ["procedure_key"],
    ]:
        if all(c in left.columns and c in right.columns for c in cols):
            left_nonempty = left[cols[0]].fillna("").astype(str).str.strip().ne("").any()
            right_nonempty = right[cols[0]].fillna("").astype(str).str.strip().ne("").any()
            if left_nonempty and right_nonempty:
                return cols
    return []

if df_doc_meta.empty:
    doc_case_agg = pd.DataFrame(columns=[
        "procedureId", "affId", "internal_key", "procedure_key",
        "available_document_types", "relevant_document_types",
        "available_document_dates", "relevant_document_dates",
        "total_document_count", "relevant_document_count", "irrelevant_document_count",
        "case_relevance_flag",
    ])
else:
    df_doc_meta["relevance_flag"] = bool_series(df_doc_meta["relevance_flag"]) if "relevance_flag" in df_doc_meta.columns else False
    if (
        "procedureId" in df_doc_meta.columns
        and df_doc_meta["procedureId"]
            .fillna("")
            .astype(str)
            .str.strip()
            .ne("")
            .any()
    ):
        group_col = "procedureId"
    elif (
        "affId" in df_doc_meta.columns
        and df_doc_meta["affId"]
            .fillna("")
            .astype(str)
            .str.strip()
            .ne("")
            .any()
    ):
        group_col = "affId"
        print(
            "Warning: procedureId unavailable for document aggregation; "
            "falling back to affId."
        )
    else:
        group_col = next(
            (
                c
                for c in ["internal_key", "procedure_key"]
                if c in df_doc_meta.columns
            ),
            "procedure_key",
        )
    count_col = "document_dedup_key" if "document_dedup_key" in df_doc_meta.columns else "logicDocId"

    agg_kwargs = {
        "available_document_types": ("normalized_docTypeCode", unique_join),
        "total_document_count": (count_col, "count"),
        "relevant_document_count": ("relevance_flag", "sum"),
    }
    if "docDate" in df_doc_meta.columns:
        agg_kwargs["available_document_dates"] = ("docDate", normalized_date_join)

    doc_case_agg = df_doc_meta.groupby(group_col, dropna=False).agg(**agg_kwargs).reset_index()

    # Add representative identifiers for easier downstream joins/inspection.
    id_cols = [c for c in ["procedureId", "affId", "internal_key", "procedure_key", "procedure_key_base"] if c in df_doc_meta.columns and c != group_col]
    if id_cols:
        id_rep = df_doc_meta.groupby(group_col, dropna=False)[id_cols].agg(unique_join).reset_index()
        doc_case_agg = doc_case_agg.merge(id_rep, on=group_col, how="left")

    rel_doc = df_doc_meta[df_doc_meta["relevance_flag"]].copy()
    rel_types = rel_doc.groupby(group_col, dropna=False)["normalized_docTypeCode"].apply(unique_join).reset_index(name="relevant_document_types")
    doc_case_agg = doc_case_agg.merge(rel_types, on=group_col, how="left")

    if "docDate" in df_doc_meta.columns:
        rel_dates = rel_doc.groupby(group_col, dropna=False)["docDate"].apply(normalized_date_join).reset_index(name="relevant_document_dates")
        doc_case_agg = doc_case_agg.merge(rel_dates, on=group_col, how="left")

    for col in ["relevant_document_types", "available_document_dates", "relevant_document_dates"]:
        if col not in doc_case_agg.columns:
            doc_case_agg[col] = ""
        doc_case_agg[col] = doc_case_agg[col].fillna("")

    doc_case_agg["irrelevant_document_count"] = doc_case_agg["total_document_count"] - doc_case_agg["relevant_document_count"]
    doc_case_agg["case_relevance_flag"] = doc_case_agg["relevant_document_count"].gt(0)

# Start case manifest with full InfoCuria case skeleton, then left-join procedural metadata and document availability.
df_case_metadata_manifest = df_cases.copy()

# Join flattened procedural metadata by stable InfoCuria identifiers, not only by reconstructed keys.
metadata_join_keys = best_join_key_cols(df_case_metadata_manifest, df_case_flat)
if not df_case_flat.empty and metadata_join_keys:
    metadata_cols = [c for c in df_case_flat.columns if c not in df_case_metadata_manifest.columns or c in metadata_join_keys]
    df_case_metadata_manifest = df_case_metadata_manifest.merge(
        df_case_flat[metadata_cols].drop_duplicates(metadata_join_keys, keep="last"),
        on=metadata_join_keys,
        how="left",
        suffixes=("", "__metadata"),
    )
else:
    df_case_metadata_manifest["metadata_join_status"] = "no_raw_metadata_jsonl_records_or_no_join_key"

if "metadata_join_status" not in df_case_metadata_manifest.columns:
    joined_indicator_col = next((c for c in ["totalHits", "publishedAffId", "jurisdictionCode", "numDocs"] if c in df_case_metadata_manifest.columns), None)
    metadata_joined = df_case_metadata_manifest[joined_indicator_col].notna() if joined_indicator_col else pd.Series(False, index=df_case_metadata_manifest.index)
    df_case_metadata_manifest["metadata_join_status"] = metadata_joined.map({True: "metadata_joined", False: "no_metadata_join"})

# Join document aggregation by procedureId first. affId is only a fallback.
doc_join_keys = best_join_key_cols(df_case_metadata_manifest, doc_case_agg)
if not doc_case_agg.empty and doc_join_keys:
    doc_agg_cols = [c for c in doc_case_agg.columns if c not in df_case_metadata_manifest.columns or c in doc_join_keys]
    df_case_metadata_manifest = df_case_metadata_manifest.merge(doc_case_agg[doc_agg_cols], on=doc_join_keys, how="left")
else:
    print("Warning: no suitable key found for document aggregation join.")

for col in ["available_document_types", "relevant_document_types", "available_document_dates", "relevant_document_dates"]:
    if col not in df_case_metadata_manifest.columns:
        df_case_metadata_manifest[col] = ""
    df_case_metadata_manifest[col] = df_case_metadata_manifest[col].fillna("")

for col in ["total_document_count", "relevant_document_count", "irrelevant_document_count"]:
    if col not in df_case_metadata_manifest.columns:
        df_case_metadata_manifest[col] = 0
    df_case_metadata_manifest[col] = pd.to_numeric(df_case_metadata_manifest[col], errors="coerce").fillna(0).astype(int)

df_case_metadata_manifest["case_relevance_flag"] = df_case_metadata_manifest["relevant_document_count"].gt(0)
df_case_metadata_manifest = df_case_metadata_manifest.drop(columns=[c for c in DROP_NON_ENGLISH_METADATA_COLUMNS if c in df_case_metadata_manifest.columns], errors="ignore")

print("Case metadata manifest rows:", len(df_case_metadata_manifest))
print("Metadata join keys used:", metadata_join_keys)
print("Document aggregation join keys used:", doc_join_keys)
display(df_case_metadata_manifest[[
    c for c in ["internal_key", "procedure_key", "procedure_key_base", "publishedId", "affId", "procedureId", "available_document_types", "relevant_document_types", "available_document_dates", "relevant_document_dates", "total_document_count", "relevant_document_count", "irrelevant_document_count", "case_relevance_flag"]
    if c in df_case_metadata_manifest.columns
]].head(20))


Streaming flatten raw JSONL: 0it [00:00, ?it/s]

Streamed case records: 57,413
Streamed document records: 154,695
Flatten failures: none
Case metadata rows after dedupe: (57413, 68)
Document metadata rows after dedupe, without context text: (154695, 44)


,internal_key,procedure_key,procedure_key_base,infocuria_slash_procedure_key,infocuria_slash_procedure_key_base,case_number_raw,case_number_clean,case_prefix,case_number_int,case_number_int_full,case_year_raw,case_year_full,case_suffix,affId,publishedId,publishedAffId,procedureId,jurisdiction,jurisdictionCode,nature,natureCode,procType,procNumber,suiteNumber,case_name_en,case_object_en,introductionDate,procIntroDate,introductionYear,closeDate,procClosDate,closeYear,affairState,affairStateCode,procState,procLang,joinExist,pilot,idPilot,pilot_case_publishedId,joinAffairs,joinAffairs_publishedIds,pourvoiAffIds,pourvoi_publishedIds,originPourvoiIds,origin_pourvoi_publishedIds,reExamenAffIds,originReExamanIds,transfertAffIds,originTransfertIds,matCode,subject_matter_en,reportingJudge,formation,advocate_general_code,resultTypes,procedure_result_en,parties,conclsDate,conclLang,delivery_dates,delivery_types,numDocs,doctrineNotes_count,lastModifiedDate,totalHits,inner_document_totalHits,raw_case_content_json
0,2000/0001/C//,2000/0001/C//,2000/0001/C/,2000/0001/C//,2000/0001/C/,C-1/00,C-1/00,C,1,0001,00,2000,,C/0001/00/00000000RD/01,C-1/00,C-1/00,C/0001/00/00000000RD/01/P/01,Cour de justice,C,Recours direct,RD,P,1,1,Commission v France,"Failure by a State to comply with its obligations ─ Refusal to adopt the necessary measures to comply with Council Decision 98/256/EC (emergency measures to protect against bovine spongiform encephalopathy) and Commission Decision 1999/514/EC (setting the date for resumption of shipments of ""DBE...",2000-01-04,NaN,2000,2001-12-13,2001-12-13,2001.0,Affaire clôturée,CLOTPUB,CLOTPUB,FR,0,0,NaN,,[],[],[],[],[],[],[],[],[],[],"[""AGRI"", ""AGRI.VETE"", ""AGRI.OCMA""]",Agriculture|Common organisation of the markets|Veterinary legislation,SEV,CP_C,MIS,"[""CONS=OB"", ""CONS=RF""]",Action for a declaration of failure to fulfil obligations: application granted|Action for a declaration of failure to fulfil obligations: dismissal on substantive grounds,Commission / France,"[""2001-09-20""]",FR,"[""2001-06-19"", ""2001-12-13""]","[""ARRET"", ""PO""]",C2000/0001/J,7,2026-09-17T08:25:52.335,1,5,
1,2000/0001/C/SA/,2000/0001/C/SA/,2000/0001/C/,2000/0001/C/SA/,2000/0001/C/,C-1/00 SA,C-1/00 SA,C,1,0001,00,2000,SA,C/0001/00/00000000SA/01,C-1/00 SA,C-1/00 SA,C/0001/00/00000000SA/01/P/01,Cour de justice,C,Saisie-arrêt,SA,P,1,1,Cotecna Inspection v Commission,"Application for leave pursuant to Article 1 of the Protocol on Privileges and Immunities of the European Communities to attach assets held by the Commission, by way of enforcement of an arbitration award made against the State of Djibouti which has been rendered enforceable by order of a Belgian...",2000-12-14,2000-12-14,2000,2001-05-29,2001-05-29,2001.0,Affaire clôturée,CLOTPUB,NaN,FR,0,0,NaN,,[],[],[],[],[],[],[],[],[],[],"[""PRIV""]",Privileges and immunities,VBO,CP_C,RUI,"[""IMMU=RF""]",Request concerning immunities: dismissal on substantive grounds,Cotecna Inspection / Commission,[],,[],[],C2000/0001/S,0,2025-10-25T15:04:50.623,1,2,
2,2001/0001/C/P/,2001/0001/C/P/,2001/0001/C/,2001/0001/C/P/,2001/0001/C/,C-1/01 P,C-1/01 P,C,1,0001,01,2001,P,C/0001/01/00000000PV/01,C-1/01 P,C-1/01 P,C/0001/01/00000000PV/01/P/01,Cour de justice,C,Pourvoi,PV,P,1,1,Asia Motor France and Others v Commission,Case C-01/01 P\r\n\r\nAppeal against the judgment in Case T-154/98 by which the Court of First Instance dismissed an application for annulment of a decision rejecting the appellants' complaints regarding an alleged agreement between importers into France of five makes of Japanese motor-car.,2001-01-03,2001-01-03,2001,2001-09-20,2001-09-20,2001.0,Affaire clôturée,CLOTPUB,CLOTPUB,FR,0,0,NaN,,[],[],"[""C/0001/01/00000000PV/01/P/01#C-1/01 P""]","[""C-1/01 P""]","[""T/0154/98/00000000RD/01/P/01#T-154/98""]","[""T-154/98""]",[],[],[],[],"[""CONC"", ""CONC.ENTE""]","Agreements, decisions and concerted practices|Competition",STG,II_C,JAC,"[""ANNU"", ""PVOI=RI"", ""PVOI=RF""]",Actions for ann

,internal_key,procedure_key,procedure_key_base,infocuria_slash_procedure_key,infocuria_slash_procedure_key_base,case_number_raw,case_number_clean,case_prefix,case_number_int,case_number_int_full,case_year_full,case_suffix,affId,publishedId,procedureId,procLang,case_name_en,introductionDate,closeDate,procIntroDate,procClosDate,doc_position,inner_hit_id,docId,logicDocId,jpLogicDocId,idProcedure,idPublished,docType,docTypeCode,normalized_docTypeCode,relevance_flag,relevance_category,relevance_reason,lnsDec,numDoc,docDate,docFormat,docFormats,docLang,available_languages,docUrl,contentLength,document_dedup_key
0,2000/0001/C//,2000/0001/C//,2000/0001/C/,2000/0001/C//,2000/0001/C/,C-1/00,C-1/00,C,1,0001,2000,,C/0001/00/00000000RD/01,C-1/00,C/0001/00/00000000RD/01/P/01,FR,Commission v France,2000-01-04,2001-12-13,NaN,2001-12-13,0,C/0001/00/00000000RD/01/P/01-623390,623390,id_53538,id_null,C/0001/00/00000000RD/01/P/01,C-1/00,Avis (JO),AVI_COMM,AVI_COMM,False,irrelevant,irrelevant_non_final_or_duplicate_or_administrative_document,,,2002-06-14,HTML,"[""HTML""]",FR,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]",,0,C/0001/00/00000000RD/01/P/01|id_53538|AVI_COMM|2002-06-14
1,2000/0001/C//,2000/0001/C//,2000/0001/C/,2000/0001/C//,2000/0001/C/,C-1/00,C-1/00,C,1,0001,2000,,C/0001/00/00000000RD/01,C-1/00,C/0001/00/00000000RD/01/P/01,FR,Commission v France,2000-01-04,2001-12-13,NaN,2001-12-13,1,C/0001/00/00000000RD/01/P/01-1216021,1216021,id_85906,id_null,C/0001/00/00000000RD/01/P/01,C-1/00,Arrêt (Sommaire),ARRET_SOM,ARRET_SOM,True,relevant,relevant_judgment_summary,,C2000/0001/J,2001-12-13,PDF,"[""PDF""]",FR,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]",,0,C/0001/00/00000000RD/01/P/01|id_85906|ARRET_SOM|2001-12-13
2,2000/0001/C//,2000/0001/C//,2000/0001/C/,2000/0001/C//,2000/0001/C/,C-1/00,C-1/00,C,1,0001,2000,,C/0001/00/00000000RD/01,C-1/00,C/0001/00/00000000RD/01/P/01,FR,Commission v France,2000-01-04,2001-12-13,NaN,2001-12-13,2,C/0001/00/00000000RD/01/P/01-616977,616977,id_46950,id_null,C/0001/00/00000000RD/01/P/01,C-1/00,Arrêt,ARRET,ARRET,True,relevant,relevant_judgment,217582,C2000/0001/J,2001-12-13,HTML,"[""HTML"", ""PDF""]",FR,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]",,0,C/0001/00/00000000RD/01/P/01|id_46950|ARRET|2001-12-13


Document metadata base rows: 154695


,internal_key,procedure_key,procedure_key_base,infocuria_slash_procedure_key,infocuria_slash_procedure_key_base,case_number_raw,case_number_clean,case_prefix,case_number_int,case_number_int_full,case_year_full,case_suffix,affId,publishedId,procedureId,procLang,case_name_en,introductionDate,closeDate,procIntroDate,procClosDate,doc_position,inner_hit_id,docId,logicDocId,jpLogicDocId,idProcedure,idPublished,docType,docTypeCode,normalized_docTypeCode,relevance_flag,relevance_category,relevance_reason,lnsDec,numDoc,docDate,docFormat,docFormats,docLang,available_languages,docUrl,contentLength,document_dedup_key
0,2000/0001/C//,2000/0001/C//,2000/0001/C/,2000/0001/C//,2000/0001/C/,C-1/00,C-1/00,C,1,0001,2000,,C/0001/00/00000000RD/01,C-1/00,C/0001/00/00000000RD/01/P/01,FR,Commission v France,2000-01-04,2001-12-13,NaN,2001-12-13,0,C/0001/00/00000000RD/01/P/01-623390,623390,id_53538,id_null,C/0001/00/00000000RD/01/P/01,C-1/00,Avis (JO),AVI_COMM,AVI_COMM,False,irrelevant,irrelevant_non_final_or_duplicate_or_administrative_document,,,2002-06-14,HTML,"[""HTML""]",FR,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]",,0,C/0001/00/00000000RD/01/P/01|id_53538|AVI_COMM|2002-06-14
1,2000/0001/C//,2000/0001/C//,2000/0001/C/,2000/0001/C//,2000/0001/C/,C-1/00,C-1/00,C,1,0001,2000,,C/0001/00/00000000RD/01,C-1/00,C/0001/00/00000000RD/01/P/01,FR,Commission v France,2000-01-04,2001-12-13,NaN,2001-12-13,1,C/0001/00/00000000RD/01/P/01-1216021,1216021,id_85906,id_null,C/0001/00/00000000RD/01/P/01,C-1/00,Arrêt (Sommaire),ARRET_SOM,ARRET_SOM,True,relevant,relevant_judgment_summary,,C2000/0001/J,2001-12-13,PDF,"[""PDF""]",FR,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]",,0,C/0001/00/00000000RD/01/P/01|id_85906|ARRET_SOM|2001-12-13
2,2000/0001/C//,2000/0001/C//,2000/0001/C/,2000/0001/C//,2000/0001/C/,C-1/00,C-1/00,C,1,0001,2000,,C/0001/00/00000000RD/01,C-1/00,C/0001/00/00000000RD/01/P/01,FR,Commission v France,2000-01-04,2001-12-13,NaN,2001-12-13,2,C/0001/00/00000000RD/01/P/01-616977,616977,id_46950,id_null,C/0001/00/00000000RD/01/P/01,C-1/00,Arrêt,ARRET,ARRET,True,relevant,relevant_judgment,217582,C2000/0001/J,2001-12-13,HTML,"[""HTML"", ""PDF""]",FR,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]",,0,C/0001/00/00000000RD/01/P/01|id_46950|ARRET|2001-12-13
3,2000/0001/C//,2000/0001/C//,2000/0001/C/,2000/0001/C//,2000/0001/C/,C-1/00,C-1/00,C,1,0001,2000,,C/0001/00/00000000RD/01,C-1/00,C/0001/00/00000000RD/01/P/01,FR,Commission v France,2000-01-04,2001-12-13,NaN,2001-12-13,3,C/0001/00/00000000RD/01/P/01-616653,616653,id_46615,id_null,C/0001/00/00000000RD/01/P/01,C-1/00,Conclusions,CONCL,CONCL,False,irrelevant,irrelevant_non_final_or_duplicate_or_administrative_document,,C2000/0001/J,2001-09-20,HTML,"[""HTML"", ""PDF""]",FR,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]",,0,C/0001/00/00000000RD/01/P/01|id_46615|CONCL|2001-09-20
4,2000/0001/C//,2000/0001/C//,2000/0001/C/,2000/0001/C//,2000/0001/C/,C-1/00,C-1/00,C,1,0001,2000,,C/0001/00/00000000RD/01,C-1/00,C/0001/00/00000000RD/01/P/01,FR,Commission v France,2000-01-04,2001-12-13,NaN,2001-12-13,4,C/0001/00/00000000RD/01/P/01-623384,623384,id_53532,id_null,C/0001/00/00000000RD/01/P/01,C-1/00,Arrêt (JO),ARR_COMM,ARR_COMM,False,irrelevant,irrelevant_non_final_or_duplicate_or_administrative_document,,,2002-02-16,HTML,"[""HTML""]",FR,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]",,0,C/0001/00/00000000RD/01/P/01|id_53532|ARR_COMM|2002-02-16
5,2000/0001/C/SA/,2000/0001/C/SA/,2000/0001/C/,2000/0001/C/SA/,2000/0001/C/,C-1/00 SA,C-1/00 SA,C,1,0001,2000,SA,C/0001/00/00000000SA/01,C-1/00 SA,C/0001/00/00000000SA/01/P/01,FR,Cotecna Inspection v Commission,2000-12-14,2001-05-29,2000-12-14,2001-05-29,0,C/0001/00/00000000SA/01/P/01-616472,616472,id_46421,NaN,C/0001/00/00000000SA/01/P/01,C-1/00 SA,Ordonnance,ORD,ORD,Fals

Relevant document rows: 42847


,procedure_key,procedure_key_base,infocuria_slash_procedure_key,infocuria_slash_procedure_key_base,publishedId,logicDocId,docType,docTypeCode,docDate,available_languages,docFormats,relevance_reason
0,2000/0001/C//,2000/0001/C/,2000/0001/C//,2000/0001/C/,C-1/00,id_85906,Arrêt (Sommaire),ARRET_SOM,2001-12-13,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]","[""PDF""]",relevant_judgment_summary
1,2000/0001/C//,2000/0001/C/,2000/0001/C//,2000/0001/C/,C-1/00,id_46950,Arrêt,ARRET,2001-12-13,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]","[""HTML"", ""PDF""]",relevant_judgment
2,2002/0001/C//,2002/0001/C/,2002/0001/C//,2002/0001/C/,C-1/02,id_49061,Arrêt,ARRET,2004-04-01,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]","[""HTML"", ""PDF""]",relevant_judgment
3,2002/0001/C//,2002/0001/C/,2002/0001/C//,2002/0001/C/,C-1/02,id_55247,Arrêt (Sommaire),ARRET_SOM,2004-04-01,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]","[""HTML"", ""PDF""]",relevant_judgment_summary
4,2003/0001/C//,2003/0001/C/,2003/0001/C//,2003/0001/C/,C-1/03,id_65336,Arrêt (Sommaire),ARRET_SOM,2004-09-07,"[""CS"", ""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""ET"", ""FI"", ""FR"", ""HU"", ""IT"", ""LT"", ""LV"", ""NL"", ""PL"", ""PT"", ""SK"", ""SL"", ""SV""]","[""HTML"", ""PDF""]",relevant_judgment_summary
5,2003/0001/C//,2003/0001/C/,2003/0001/C//,2003/0001/C/,C-1/03,id_49453,Arrêt,ARRET,2004-09-07,"[""CS"", ""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""ET"", ""FI"", ""FR"", ""HU"", ""IT"", ""LT"", ""LV"", ""MT"", ""NL"", ""PL"", ""PT"", ""SK"", ""SL"", ""SV""]","[""HTML"", ""PDF""]",relevant_judgment
6,2004/0001/C//,2004/0001/C/,2004/0001/C//,2004/0001/C/,C-1/04,id_57812,Arrêt,ARRET,2006-01-17,"[""CS"", ""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""ET"", ""FI"", ""FR"", ""HU"", ""IT"", ""LT"", ""LV"", ""MT"", ""NL"", ""PL"", ""PT"", ""SK"", ""SL"", ""SV""]","[""HTML"", ""PDF""]",relevant_judgment
7,2004/0001/C//,2004/0001/C/,2004/0001/C//,2004/0001/C/,C-1/04,id_64087,Arrêt (Sommaire),ARRET_SOM,2006-01-17,"[""CS"", ""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""ET"", ""FI"", ""FR"", ""HU"", ""IT"", ""LT"", ""LV"", ""NL"", ""PL"", ""PT"", ""SK"", ""SL"", ""SV""]","[""HTML"", ""PDF""]",relevant_judgment_summary
8,2005/0001/C//,2005/0001/C/,2005/0001/C//,2005/0001/C/,C-1/05,id_61266,Arrêt (Sommaire),ARRET_SOM,2007-01-09,"[""BG"", ""CS"", ""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""ET"", ""FI"", ""FR"", ""HU"", ""IT"", ""LT"", ""LV"", ""NL"", ""PL"", ""PT"", ""RO"", ""SK"", ""SL"", ""SV""]","[""HTML"", ""PDF""]",relevant_judgment_summary
9,2005/0001/C//,2005/0001/C/,2005/0001/C//,2005/0001/C/,C-1/05,id_64718,Arrêt,ARRET,2007-01-09,"[""BG"", ""CS"", ""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""ET"", ""FI"", ""FR"", ""HU"", ""IT"", ""LT"", ""LV"", ""MT"", ""NL"", ""PL"", ""PT"", ""RO"", ""SK"", ""SL"", ""SV""]","[""HTML"", ""PDF""]",relevant_judgment


Case metadata manifest rows: 57413
Metadata join keys used: ['procedureId']
Document aggregation join keys used: ['procedureId']


,internal_key,procedure_key,procedure_key_base,publishedId,affId,procedureId,available_document_types,relevant_document_types,available_document_dates,relevant_document_dates,total_document_count,relevant_document_count,irrelevant_document_count,case_relevance_flag
0,2000/0001/C//,2000/0001/C//,2000/0001/C/,C-1/00,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET|ARRET_SOM|ARR_COMM|AVI_COMM|CONCL,ARRET|ARRET_SOM,2001-09-20|2001-12-13|2002-02-16|2002-06-14,2001-12-13,5,2,3,True
1,2000/0001/C/SA/,2000/0001/C/SA/,2000/0001/C/,C-1/00 SA,C/0001/00/00000000SA/01,C/0001/00/00000000SA/01/P/01,ORD|ORD_SOM,,2001-05-29,,2,0,2,False
2,2001/0001/C/P/,2001/0001/C/P/,2001/0001/C/,C-1/01 P,C/0001/01/00000000PV/01,C/0001/01/00000000PV/01/P/01,ORD|ORD_SOM,,2001-09-20,,2,0,2,False
3,2002/0001/C//,2002/0001/C//,2002/0001/C/,C-1/02,C/0001/02/00000000RP/01,C/0001/02/00000000RP/01/P/01,ARRET|ARRET_SOM|ARR_COMM|CONCL|DDP_COMM|ORD_COMM,ARRET|ARRET_SOM,2002-03-02|2003-07-03|2003-08-22|2003-09-06|2004-04-01|2004-04-30,2004-04-01,7,2,5,True
4,2002/0001/C/SA/,2002/0001/C/SA/,2002/0001/C/,C-1/02 SA,C/0001/02/00000000SA/01,C/0001/02/00000000SA/01/P/01,ORD|ORD_SOM,,2003-03-27,,2,0,2,False
5,2003/0001/C//,2003/0001/C//,2003/0001/C/,C-1/03,C/0001/03/00000000RP/01,C/0001/03/00000000RP/01/P/01,ARRET|ARRET_SOM|ARR_COMM|CONCL|DDP_COMM|RAD_COMM|REQ_COMM,ARRET|ARRET_SOM,2003-02-07|2003-04-04|2004-01-29|2004-04-16|2004-09-07|2004-10-23,2004-09-07,7,2,5,True
6,2003/0001/C/SA/,2003/0001/C/SA/,2003/0001/C/,C-1/03 SA,C/0001/03/00000000SA/01,C/0001/03/00000000SA/01/P/01,,,,,0,0,0,False
7,2004/0001/C//,2004/0001/C//,2004/0001/C/,C-1/04,C/0001/04/00000000RP/01,C/0001/04/00000000RP/01/P/01,ARRET|ARRET_SOM|AVI_COMM|CONCL|DDP_COMM|ORD_COMM|RAD_COMM|REQ_COMM,ARRET|ARRET_SOM,2004-03-05|2004-04-26|2005-03-04|2005-03-19|2005-09-06|2006-01-17,2006-01-17,9,2,7,True
8,2004/0001/C/SA/,2004/0001/C/SA/,2004/0001/C/,C-1/04 SA,C/0001/04/00000000SA/01,C/0001/04/00000000SA/01/P/01,ORD|ORD_SOM,,2004-12-14,,2,0,2,False
9,2005/0001/C//,2005/0001/C//,2005/0001/C/,C-1/05,C/0001/05/00000000RP/01,C/0001/05/00000000RP/01/P/01,ARRET|ARRET_SOM|CONCL|DDP_COMM,ARRET|ARRET_SOM,2005-03-05|2006-04-27|2007-01-09,2007-01-09,4,2,2,True


## 6. Derive `proceeding_id` and ordinary appeal relations

### Proceedings

A proceeding is represented as a **layer on the case table**, not as a separate table. The grouping is derived from `idPilot` and `joinAffairs_publishedIds`:

- the procedure identifier embedded before `#` in `idPilot` is linked to the current `procedureId`;
- joined published case identifiers are matched exactly to the case-level published/raw case number, preserving suffixes such as `P`, `R`, etc.;
- connected components of these links define proceedings, so asymmetric joined-case metadata still produces one group;
- singleton cases receive their own proceeding ID.

The generated `proceeding_id` is deterministic: it is a short hash of the sorted member `procedureId` values. `proceeding_anchor_procedureId`, `proceeding_size`, and `proceeding_is_joined` are included for transparency.

### Appeals

Ordinary appeals are derived from `originPourvoiIds`. Each value contains a source `procedureId` before the `#`; that source is the case being appealed, while the current row's `procedureId` is the appeal case. `pourvoiAffIds` is retained as a validation field. `reExamenAffIds` / `originReExamanIds` are intentionally **not** treated as ordinary appeals.

In [6]:
# ---------------------------------------------------------------------
# Proceeding layer: connected components from idPilot + joined cases
# ---------------------------------------------------------------------
import hashlib


def parse_jsonish_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    s = clean_str(value)
    if not s:
        return []
    try:
        obj = json.loads(s)
        return obj if isinstance(obj, list) else [obj]
    except Exception:
        return [s]


def extract_procedure_id_from_ref(value):
    """Extract the InfoCuria procedureId part before '#'."""
    if isinstance(value, dict):
        for key in ["procedureId", "idProcedure", "id", "value"]:
            candidate = clean_str(value.get(key))
            if candidate:
                value = candidate
                break
    s = clean_str(value)
    if not s:
        return ""
    return s.split("#", 1)[0].strip()


def normalize_published_case(value):
    return normalize_whitespace(value).upper().replace("–", "-").replace("—", "-")


cases_struct = df_case_metadata_manifest.copy()
cases_struct["procedureId"] = cases_struct["procedureId"].fillna("").astype(str).str.strip()

# Exact published/raw identifier lookup. Suffixes are deliberately preserved.
published_lookup = {}
for _, row in cases_struct.iterrows():
    pid = clean_str(row.get("procedureId"))
    if not pid:
        continue
    for col in ["case_number_raw", "publishedId", "publishedAffId"]:
        key = normalize_published_case(row.get(col))
        if key and key not in published_lookup:
            published_lookup[key] = pid

# Simple union-find for connected case components.
parent = {pid: pid for pid in cases_struct["procedureId"] if pid}

def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(a, b):
    if not a or not b:
        return
    if a not in parent or b not in parent:
        return
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

unmatched_joined_case_refs = []
unmatched_pilot_refs = []

for _, row in cases_struct.iterrows():
    current_pid = clean_str(row.get("procedureId"))
    if not current_pid:
        continue

    # idPilot usually contains "procedureId#published case".
    pilot_pid = extract_procedure_id_from_ref(row.get("idPilot"))
    if pilot_pid:
        if pilot_pid in parent:
            union(current_pid, pilot_pid)
        else:
            unmatched_pilot_refs.append((current_pid, pilot_pid))

    # Joined cases are supplied as published identifiers; preserve suffixes.
    for joined_case in parse_jsonish_list(row.get("joinAffairs_publishedIds")):
        joined_key = normalize_published_case(joined_case)
        joined_pid = published_lookup.get(joined_key, "")
        if joined_pid:
            union(current_pid, joined_pid)
        elif joined_key:
            unmatched_joined_case_refs.append((current_pid, joined_key))

components = {}
for pid in parent:
    components.setdefault(find(pid), []).append(pid)

pid_to_proceeding = {}
pid_to_anchor = {}
pid_to_size = {}
for members in components.values():
    members = sorted(set(members))
    anchor = members[0]
    digest = hashlib.sha1("|".join(members).encode("utf-8")).hexdigest()[:12]
    proceeding_id = f"PROCEEDING_{digest}"
    for pid in members:
        pid_to_proceeding[pid] = proceeding_id
        pid_to_anchor[pid] = anchor
        pid_to_size[pid] = len(members)

cases_struct["proceeding_id"] = cases_struct["procedureId"].map(pid_to_proceeding).fillna("")
cases_struct["proceeding_anchor_procedureId"] = cases_struct["procedureId"].map(pid_to_anchor).fillna("")
cases_struct["proceeding_size"] = cases_struct["procedureId"].map(pid_to_size).fillna(1).astype(int)
cases_struct["proceeding_is_joined"] = cases_struct["proceeding_size"].gt(1)

df_case_metadata_manifest = cases_struct

print(f"Proceedings created: {df_case_metadata_manifest['proceeding_id'].nunique():,}")
print(f"Joined proceedings: {df_case_metadata_manifest.loc[df_case_metadata_manifest['proceeding_is_joined'], 'proceeding_id'].nunique():,}")
print(f"Unmatched idPilot procedure references: {len(unmatched_pilot_refs):,}")
print(f"Unmatched joined published-case references: {len(unmatched_joined_case_refs):,}")

# ---------------------------------------------------------------------
# Ordinary appeal edge table from originPourvoiIds
# ---------------------------------------------------------------------
case_number_by_pid = (
    df_case_metadata_manifest
    .drop_duplicates("procedureId")
    .set_index("procedureId")["case_number_raw"]
    .fillna("")
    .astype(str)
    .to_dict()
)
known_pids = set(case_number_by_pid)

appeal_rows = []
unmatched_appeal_origins = []

for _, row in df_case_metadata_manifest.iterrows():
    appeal_pid = clean_str(row.get("procedureId"))
    if not appeal_pid:
        continue
    appeal_case_number = clean_str(row.get("case_number_raw"))
    origin_refs = parse_jsonish_list(row.get("originPourvoiIds"))

    for raw_ref in origin_refs:
        origin_pid = extract_procedure_id_from_ref(raw_ref)
        if not origin_pid:
            continue
        if origin_pid not in known_pids:
            unmatched_appeal_origins.append((appeal_pid, origin_pid))

        appeal_rows.append({
            "origin_procedureId": origin_pid,
            "origin_case_number": case_number_by_pid.get(origin_pid, ""),
            "appeal_procedureId": appeal_pid,
            "appeal_case_number": appeal_case_number,
            "appeal_type": "appeal",
            "source_field": "originPourvoiIds",
            "source_reference": clean_str(raw_ref),
        })

df_appeals = pd.DataFrame(appeal_rows)
if df_appeals.empty:
    df_appeals = pd.DataFrame(columns=[
        "origin_procedureId", "origin_case_number",
        "appeal_procedureId", "appeal_case_number",
        "appeal_type", "source_field", "source_reference",
    ])
else:
    df_appeals = (
        df_appeals
        .drop_duplicates(subset=["origin_procedureId", "appeal_procedureId", "appeal_type"])
        .sort_values(["origin_procedureId", "appeal_procedureId"], kind="stable")
        .reset_index(drop=True)
    )

print(f"Ordinary appeal relations: {len(df_appeals):,}")
print(f"Unmatched appeal-origin procedureIds: {len(unmatched_appeal_origins):,}")
display(df_appeals.head(20))

Proceedings created: 51,205
Joined proceedings: 2,213
Unmatched idPilot procedure references: 0
Unmatched joined published-case references: 3
Ordinary appeal relations: 7,638
Unmatched appeal-origin procedureIds: 345


,origin_procedureId,origin_case_number,appeal_procedureId,appeal_case_number,appeal_type,source_field,source_reference
0,F/0001/05/00000000RF/01/P/01,,T/0404/06/00000000PV/01/P/01,T-404/06 P,appeal,originPourvoiIds,F/0001/05/00000000RF/01/P/01#F-1/05
1,F/0001/08/00000000RF/01/P/01,,T/0376/08/00000000PV/01/P/01,T-376/08 P,appeal,originPourvoiIds,F/0001/08/00000000RF/01/P/01#F-1/08
2,F/0001/10/00000000RF/01/P/01,,T/0126/11/00000000PV/01/P/01,T-126/11 P,appeal,originPourvoiIds,F/0001/10/00000000RF/01/P/01#F-1/10
3,F/0001/14/00000000RF/01/P/01,,T/0152/15/00000000PV/01/P/01,T-152/15 P,appeal,originPourvoiIds,F/0001/14/00000000RF/01/P/01#F-1/14
4,F/0001/16/00000000RF/01/P/01,,T/0705/16/00000000PV/01/P/01,T-705/16 P,appeal,originPourvoiIds,F/0001/16/00000000RF/01/P/01#F-1/16
5,F/0002/06/00000000RF/01/P/01,,T/0278/07/00000000PV/01/P/01,T-278/07 P,appeal,originPourvoiIds,F/0002/06/00000000RF/01/P/01#F-2/06
6,F/0002/10/00000000RF/01/P/01,,T/0594/10/00000000PV/01/P/01,T-594/10 P,appeal,originPourvoiIds,F/0002/10/00000000RF/01/P/01#F-2/10
7,F/0002/12/00000000RF/01/P/01,,T/0026/15/00000000PV/01/P/01,T-26/15 P,appeal,originPourvoiIds,F/0002/12/00000000RF/01/P/01#F-2/12
8,F/0002/12/00000000RF/01/P/01,,T/0027/15/00000000PV/01/P/01,T-27/15 P,appeal,originPourvoiIds,F/0002/12/00000000RF/01/P/01#F-2/12
9,F/0002/13/00000000RF/01/P/01,,T/0148/14/00000000PV/01/P/01,T-148/14 P,appeal,originPourvoiIds,F/0002/13/00000000RF/01/P/01#F-2/13


## 7. Save publication tables

Both XLSX and CSV versions are written. CSV is convenient for repository deposition and machine reuse; XLSX is retained for inspection during pipeline development. The two formats contain the same rows and columns.

In [7]:
# ---------------------------------------------------------------------
# Publication tables
# ---------------------------------------------------------------------
df_document_metadata_manifest = df_doc_meta.copy()

# Keep all document rows. Relevance is classification metadata only.
for frame in [df_case_metadata_manifest, df_document_metadata_manifest, df_appeals]:
    if "procedureId" in frame.columns:
        frame["procedureId"] = frame["procedureId"].fillna("").astype(str)

# Remove only deliberately huge/raw helper fields from the tabular deposit.
df_case_metadata_manifest = df_case_metadata_manifest.drop(
    columns=[c for c in DROP_NON_ENGLISH_METADATA_COLUMNS if c in df_case_metadata_manifest.columns],
    errors="ignore",
)
df_document_metadata_manifest = df_document_metadata_manifest.drop(
    columns=[c for c in DROP_NON_ENGLISH_METADATA_COLUMNS + ["document_dedup_key"] if c in df_document_metadata_manifest.columns],
    errors="ignore",
)

save_table(df_case_metadata_manifest, CASE_METADATA_XLSX, "cases")
save_table(df_document_metadata_manifest, DOCUMENT_METADATA_XLSX, "documents")
save_table(df_appeals, APPEAL_METADATA_XLSX, "appeals")

df_case_metadata_manifest.to_csv(CASE_METADATA_CSV, index=False)
df_document_metadata_manifest.to_csv(DOCUMENT_METADATA_CSV, index=False)
df_appeals.to_csv(APPEAL_METADATA_CSV, index=False)

print("Saved cases:", CASE_METADATA_XLSX)
print("Saved documents:", DOCUMENT_METADATA_XLSX)
print("Saved appeals:", APPEAL_METADATA_XLSX)

Saved cases: /home/edik/projects/ec_cjeu/output/court_infocuria/court_cases.xlsx
Saved documents: /home/edik/projects/ec_cjeu/output/court_infocuria/court_documents.xlsx
Saved appeals: /home/edik/projects/ec_cjeu/output/court_infocuria/court_appeals.xlsx


## 8. Technical validation summary

These checks are intended to feed directly into the Data Descriptor's technical validation: uniqueness of the case key, completeness of derived proceeding IDs, mapping success for joined-case references and appeal origins, and the distribution of document relevance classifications. Unmatched references are reported rather than silently dropped from diagnostics.

In [8]:
print(f"Case procedures: {len(df_case_metadata_manifest):,}")
print(f"Documents: {len(df_document_metadata_manifest):,}")
print(f"Appeal relations: {len(df_appeals):,}")

nonempty_pid = df_case_metadata_manifest["procedureId"].fillna("").astype(str).str.strip().ne("")
print("Duplicate non-empty procedureId rows:", int(df_case_metadata_manifest.loc[nonempty_pid, "procedureId"].duplicated().sum()))
print("Cases missing proceeding_id:", int(df_case_metadata_manifest["proceeding_id"].fillna("").eq("").sum()))
print("Unmatched idPilot refs:", len(unmatched_pilot_refs))
print("Unmatched joined refs:", len(unmatched_joined_case_refs))
print("Unmatched appeal origins:", len(unmatched_appeal_origins))

if "relevance_flag" in df_document_metadata_manifest.columns:
    rel = df_document_metadata_manifest["relevance_flag"].astype(str).str.lower().isin({"true", "1", "yes"})
    print("Relevant documents under project definition:", int(rel.sum()))
    display(
        df_document_metadata_manifest
        .groupby(["docType", "normalized_docTypeCode", "relevance_flag"], dropna=False)
        .size()
        .reset_index(name="documents")
        .sort_values("documents", ascending=False)
        .head(100)
    )

Case procedures: 57,413
Documents: 154,695
Appeal relations: 7,638
Duplicate non-empty procedureId rows: 0
Cases missing proceeding_id: 0
Unmatched idPilot refs: 0
Unmatched joined refs: 3
Unmatched appeal origins: 345
Relevant documents under project definition: 42847


,docType,normalized_docTypeCode,relevance_flag,documents
4,Arrêt (JO),ARR_COMM,False,19393
0,Arrêt,ARRET,True,19126
32,Requête (JO),REQ_COMM,False,17278
5,Arrêt (Sommaire),ARRET_SOM,True,16866
22,Ordonnance,ORD_NP,False,13381
9,Conclusions,CONCL,False,12145
10,Demande (JO),DDP_COMM,False,9898
1,Arrêt,ARRET_NP,False,7343
2,Arrêt (Information),ARRET_DR,True,6836
25,Ordonnance (JO),ORD_COMM,False,5171
